# Part 6 — Prototyping & Prompt Engineering

*CinemaStream — The Forward Deployed Engineer's Handbook*

---

In [ ]:
# ── CinemaStream: one-time setup ──────────────────────────────────────────────
# Run this cell FIRST if you are on Google Colab or a fresh local environment.
# Skip it if you have already cloned the repo and installed requirements.
#
# !pip install -r requirements.txt
# !git clone https://github.com/YOUR_ORG/cinemastream.git
# import os; os.chdir("cinemastream")
# ─────────────────────────────────────────────────────────────────────────────
# Ensure the canonical dataset exists (deterministic; safe to re-run).
try:
    from cinemastream.scripts.generate_data import generate
    generate()
except ModuleNotFoundError:
    print("Run the clone/cd lines above first (Colab), then re-run this cell.")

## Chapters in this notebook

- [Chapter 60: Streamlit Fundamentals](#chapter_60_streamlit_fundamentals)
- [Chapter 61: Backend Integration & Naive RAG](#chapter_61_backend_integration_naive_rag)
- [Chapter 62: Deployment](#chapter_62_deployment)
- [Chapter 63: User-Centered Design](#chapter_63_user_centered_design)
- [Chapter 64: Prompt Engineering Fundamentals](#chapter_64_prompt_engineering_fundamentals)
- [Chapter 65: Advanced Prompting — KV Cache, Prompt Caching, and the Cost/Latency Math](#chapter_65_advanced_prompting_kv_cache_prompt_caching_and_the_cost_latency_math)
- [Chapter 66: LLM Workflow Automation — LangChain, Hybrid RAG, and MCP Servers](#chapter_66_llm_workflow_automation_langchain_hybrid_rag_and_mcp_servers)
- [Chapter 66a: Statistics & Probability Foundations](#chapter_66a_statistics_probability_foundations)
- [Chapter 66b: Statistical Inference & Experimentation](#chapter_66b_statistical_inference_experimentation)
- [Chapter 66c: Linear Algebra, Embeddings & Optimization for ML](#chapter_66c_linear_algebra_embeddings_optimization_for_ml)

---

# Chapter 60: Streamlit Fundamentals

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas streamlit

### 2.1 The smallest possible app

In [ ]:
# app_minimal.py
import streamlit as st

st.title("Hello, CinemaStream")
st.write("This is a paragraph of text.")

name = st.text_input("What's your name?")
if name:
    st.write(f"Hello, {name}!")

```
(No terminal output beyond the Streamlit server banner —
the result renders in the browser at http://localhost:8501)

  You can now view your Streamlit app in your browser.
  Local URL: http://localhost:8501
```

### 2.2 The rerun model, made visible with a counter

In [ ]:
# Simulating what Streamlit does on each rerun.
# In a real Streamlit script, this entire function body
# is your top-level script code.

def simulate_rerun(widget_value, session_state):
    print(f"--- RERUN START (widget_value={widget_value}) ---")
    print("1. Imports run again (cheap — Python caches modules)")
    print("2. Every line of the script runs again, top to bottom")

    # st.session_state persists ACROSS reruns (we simulate with a dict)
    if "click_count" not in session_state:
        session_state["click_count"] = 0
        print("3. session_state initialized: click_count = 0")
    else:
        print(f"3. session_state already has click_count = {session_state['click_count']}")

    if widget_value == "clicked":
        session_state["click_count"] += 1
        print(f"4. Button was clicked -> click_count is now {session_state['click_count']}")
    else:
        print("4. Button was not clicked this rerun")

    print(f"--- RERUN END (click_count = {session_state['click_count']}) ---\n")


# Simulate three reruns: page load, click, click
state = {}
simulate_rerun(widget_value=None, session_state=state)       # initial page load
simulate_rerun(widget_value="clicked", session_state=state)  # user clicks button
simulate_rerun(widget_value="clicked", session_state=state)  # user clicks button again

```
--- RERUN START (widget_value=None) ---
1. Imports run again (cheap — Python caches modules)
2. Every line of the script runs again, top to bottom
3. session_state initialized: click_count = 0
4. Button was not clicked this rerun
--- RERUN END (click_count = 0) ---

--- RERUN START (widget_value=clicked) ---
1. Imports run again (cheap — Python caches modules)
2. Every line of the script runs again, top to bottom
3. session_state already has click_count = 0
4. Button was clicked -> click_count is now 1
--- RERUN END (click_count = 1) ---

--- RERUN START (widget_value=clicked) ---
1. Imports run again (cheap — Python caches modules)
2. Every line of the script runs again, top to bottom
3. session_state already has click_count = 1
4. Button was clicked -> click_count is now 2
--- RERUN END (click_count = 2) ---
```

### 2.3 Caching: why `@st.cache_data` matters

In [ ]:
import time
import functools

def cache_data(func):
    """A tiny stand-in for st.cache_data, showing the core idea:
    remember results keyed by input arguments."""
    cache = {}

    @functools.wraps(func)
    def wrapper(*args):
        if args in cache:
            print(f"  [cache HIT for {args}] -- skipping expensive work")
            return cache[args]
        print(f"  [cache MISS for {args}] -- doing expensive work")
        result = func(*args)
        cache[args] = result
        return result

    return wrapper


@cache_data
def load_big_dataset(n_rows: int):
    time.sleep(0.01)  # pretend this is a slow disk read
    return [f"row_{i}" for i in range(n_rows)]


# Simulate three reruns of a script that calls load_big_dataset(100)
print("Rerun 1 (page load):")
data = load_big_dataset(100)
print(f"  got {len(data)} rows")

print("Rerun 2 (user clicked an unrelated button):")
data = load_big_dataset(100)
print(f"  got {len(data)} rows")

print("Rerun 3 (user changed n_rows to 200):")
data = load_big_dataset(200)
print(f"  got {len(data)} rows")

```
Rerun 1 (page load):
  [cache MISS for (100,)] -- doing expensive work
  got 100 rows
Rerun 2 (user clicked an unrelated button):
  [cache HIT for (100,)] -- skipping expensive work
  got 100 rows
Rerun 3 (user changed n_rows to 200):
  [cache MISS for (200,)] -- doing expensive work
  got 200 rows
```

### 2.4 Layout primitives: columns, sidebar, tabs

In [ ]:
# Layout code (runs inside a real Streamlit script)
layout_plan = {
    "st.sidebar": ["filters: country multiselect, plan multiselect, reset button"],
    "st.columns(4)": ["KPI 1: total users", "KPI 2: churn rate",
                       "KPI 3: MRR", "KPI 4: total watch minutes"],
    "st.columns(2)": ["left: bar chart of genre minutes",
                       "right: bar chart of plan mix"],
    "st.dataframe": ["drill-down table of recent watch events"],
}

for region, contents in layout_plan.items():
    print(f"{region}:")
    for item in contents:
        print(f"  - {item}")

```
st.sidebar:
  - filters: country multiselect, plan multiselect, reset button
st.columns(4):
  - KPI 1: total users
  - KPI 2: churn rate
  - KPI 3: MRR
  - KPI 4: total watch minutes
st.columns(2):
  - left: bar chart of genre minutes
  - right: bar chart of plan mix
st.dataframe:
  - drill-down table of recent watch events
```

In [ ]:
# Real Streamlit layout syntax (for reference -- run via `streamlit run`)
import streamlit as st

st.sidebar.header("Filters")
country = st.sidebar.multiselect("Country", options=["SG", "MY", "ID"])

col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Users", "100")
col2.metric("Churn Rate", "24.0%")
col3.metric("MRR", "S$363.50")
col4.metric("Watch Minutes", "38,072")

left, right = st.columns(2)
with left:
    st.subheader("Genre breakdown")
    # st.bar_chart(...) goes here
with right:
    st.subheader("Plan mix")
    # st.bar_chart(...) goes here

```
(Renders in browser as: a sidebar with a country filter,
a row of 4 metric cards, then two side-by-side chart panels)
```

## 3. CinemaStream in Practice

In [ ]:
"""
CinemaStream "Ask Anything" dashboard -- Chapter 60 MVP.
Run with: streamlit run cinemastream/streamlit_app/app.py
"""
import numpy as np
import pandas as pd
import streamlit as st

COUNTRIES = ["SG", "MY", "ID", "PH", "TH", "VN", "IN"]
GENRES = ["Action", "Drama", "Comedy", "Documentary", "Thriller", "Romance"]
LANG_BY_COUNTRY = {"SG": "en", "MY": "ms", "ID": "id", "PH": "tl", "TH": "th", "VN": "vi", "IN": "hi"}
PLAN_PRICE_SGD = {"Free": 0.0, "Basic": 8.90, "Premium": 12.90}
DEVICES = ["Mobile", "TV", "Web", "Tablet"]


@st.cache_data
def make_users_df(seed: int = 42) -> pd.DataFrame:
    """100 users: 56 Free, 28 Basic, 16 Premium (canonical bible split)."""
    rng = np.random.default_rng(seed)
    plans = ["Free"] * 56 + ["Basic"] * 28 + ["Premium"] * 16
    plans.remove("Premium")  # Ravi (user_id=1)
    plans.remove("Basic")    # Siti (user_id=2)
    plans.remove("Free")     # Minh (user_id=3)
    rng.shuffle(plans)

    rows = [
        (1, "Ravi Kumar", "ravi@example.com", "IN", "hi", "Premium", "2024-01-15", False),
        (2, "Siti Rahman", "siti@example.com", "MY", "ms", "Basic", "2024-02-03", False),
        (3, "Nguyen Van Minh", "minh@example.com", "VN", "vi", "Free", "2024-03-22", True),
    ]
    for idx, plan in enumerate(plans, start=4):
        country = COUNTRIES[(idx * 3) % len(COUNTRIES)]
        signup = pd.Timestamp("2023-06-01") + pd.Timedelta(days=int(rng.integers(0, 500)))
        churned = bool(rng.random() < 0.18)
        rows.append((idx, f"User{idx:03d}", f"user{idx:03d}@example.com", country,
                      LANG_BY_COUNTRY[country], plan, signup.strftime("%Y-%m-%d"), churned))
    return pd.DataFrame(rows, columns=["user_id", "name", "email", "country", "language_pref",
                                         "plan", "signup_date", "churned"])


@st.cache_data
def make_movies_df(n_movies: int = 30) -> pd.DataFrame:
    rows = [
        (101, "Monsoon Heart", "hi", "Drama", 2024, 132, "A monsoon love story set in coastal Kerala."),
        (102, "Hujan di Singapura", "ms", "Thriller", 2023, 118, "A Singapore cybercrime unit races a deadline."),
        (103, "Office Hari Ini", "id", "Comedy", 2024, 96, "A Jakarta ad agency adjusts to permanent remote work."),
    ]
    titles_pool = ["Last Train to Hanoi", "Manila Nights", "Bangkok Static", "Quiet Tides",
        "The Durian Detective", "Songs from Sabah", "Curry & Code", "Mekong Drift",
        "Neon Temple", "Paper Boats", "The Last Recipe", "Kerala Skies",
        "Borneo Signal", "Cafe Aroy", "The Understudy", "Highway 19",
        "Glasshouse", "Echoes of Penang", "Midnight Ferry", "The Negotiator",
        "Salt and Monsoon", "Small Island Diary", "Sea of Memory", "Lagu Senja",
        "Cermin Retak", "The Long Layover", "Pulau", "Static Lines"]
    for i, title in enumerate(titles_pool[: n_movies - 3], start=104):
        genre = GENRES[i % len(GENRES)]
        lang = list(LANG_BY_COUNTRY.values())[i % len(LANG_BY_COUNTRY)]
        year = 2018 + (i % 7)
        runtime = 85 + (i * 3) % 60
        rows.append((i, title, lang, genre, year, runtime, f"{title} -- a CinemaStream original."))
    return pd.DataFrame(rows, columns=["movie_id", "title", "original_lang", "genre",
                                         "release_year", "runtime_min", "description"])


@st.cache_data
def make_watch_events_df(users_df, movies_df, n_events: int = 381, seed: int = 99) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    user_ids = users_df["user_id"].to_numpy()
    movie_ids = movies_df["movie_id"].to_numpy()
    runtime_map = movies_df.set_index("movie_id")["runtime_min"]
    country_map = users_df.set_index("user_id")["country"]

    rows = [
        (1, 1, 101, "2024-04-01 14:45:00", 132, True, "TV", "IN"),
        (2, 1, 102, "2024-04-02 17:00:00", 45, False, "Mobile", "IN"),
        (3, 2, 103, "2024-04-03 11:30:00", 96, True, "TV", "MY"),
    ]
    base = pd.Timestamp("2024-04-01")
    for event_id in range(4, n_events + 1):
        uid = int(rng.choice(user_ids))
        mid = int(rng.choice(movie_ids))
        runtime = int(runtime_map[mid])
        ts = base + pd.Timedelta(days=int(rng.integers(0, 75)),
                                  hours=int(rng.integers(0, 24)),
                                  minutes=int(rng.integers(0, 60)))
        completed = bool(rng.random() < 0.55)
        watch_minutes = runtime if completed else int(rng.integers(5, max(6, runtime - 5)))
        device = DEVICES[int(rng.integers(0, len(DEVICES)))]
        country = country_map[uid]
        rows.append((event_id, uid, mid, ts.strftime("%Y-%m-%d %H:%M:%S"),
                      watch_minutes, completed, device, country))
    return pd.DataFrame(rows, columns=["event_id", "user_id", "movie_id", "watch_started",
                                         "watch_minutes", "completed", "device", "country"])


@st.cache_data
def load_data():
    users_df = make_users_df()
    movies_df = make_movies_df()
    events_df = make_watch_events_df(users_df, movies_df)
    return users_df, movies_df, events_df


st.set_page_config(page_title="CinemaStream Analytics", page_icon=":bar_chart:", layout="wide")
st.title("CinemaStream Analytics Dashboard")
st.caption("Internal tool -- prototype built with Wei Lin, Module 8")

users_df, movies_df, events_df = load_data()

# Sidebar filters
st.sidebar.header("Filters")
if "selected_countries" not in st.session_state:
    st.session_state.selected_countries = COUNTRIES.copy()

selected_countries = st.sidebar.multiselect(
    "Country", options=COUNTRIES, default=st.session_state.selected_countries,
    key="selected_countries",
)
selected_plans = st.sidebar.multiselect(
    "Plan", options=["Free", "Basic", "Premium"], default=["Free", "Basic", "Premium"],
)
if st.sidebar.button("Reset filters"):
    st.session_state.selected_countries = COUNTRIES.copy()
    st.rerun()

active_countries = selected_countries or COUNTRIES
active_plans = selected_plans or ["Free", "Basic", "Premium"]

filtered_users = users_df[
    users_df["country"].isin(active_countries) & users_df["plan"].isin(active_plans)
]
filtered_user_ids = set(filtered_users["user_id"])
filtered_events = events_df[events_df["user_id"].isin(filtered_user_ids)]

# KPI row
col1, col2, col3, col4 = st.columns(4)
total_users = len(filtered_users)
churn_rate = filtered_users["churned"].mean() * 100 if total_users else 0.0
mrr = filtered_users.loc[~filtered_users["churned"], "plan"].map(PLAN_PRICE_SGD).sum()
total_watch_minutes = filtered_events["watch_minutes"].sum()

col1.metric("Users (filtered)", f"{total_users:,}")
col2.metric("Churn rate", f"{churn_rate:.1f}%")
col3.metric("MRR (S$, filtered)", f"S${mrr:,.2f}")
col4.metric("Total watch minutes", f"{total_watch_minutes:,}")

st.divider()

# Charts
left, right = st.columns(2)
with left:
    st.subheader("Watch minutes by genre")
    genre_minutes = (filtered_events.merge(movies_df[["movie_id", "genre"]], on="movie_id")
                     .groupby("genre")["watch_minutes"].sum().sort_values(ascending=False))
    st.bar_chart(genre_minutes)
with right:
    st.subheader("Plan mix (filtered users)")
    st.bar_chart(filtered_users["plan"].value_counts())

st.divider()

# Drill-down table
st.subheader("Recent watch events (filtered)")
show_completed_only = st.checkbox("Completed sessions only", value=False)
# filtered_events already carries `country`; only pull name + plan from users_df
# (merging a second `country` would create country_x/country_y and drop plain `country`)
table = (filtered_events.merge(movies_df[["movie_id", "title", "genre"]], on="movie_id")
         .merge(users_df[["user_id", "name", "plan"]], on="user_id"))
if show_completed_only:
    table = table[table["completed"]]
table = table.sort_values("watch_started", ascending=False).head(20)
st.dataframe(
    table[["watch_started", "name", "country", "plan", "title", "genre",
           "watch_minutes", "completed", "device"]],
    use_container_width=True, hide_index=True,
)

In [ ]:
# What the KPI cards show with no filters applied (computed directly,
# matching the logic inside app.py -- including the signup_date draw,
# which affects the random sequence used for `churned`)
import numpy as np
import pandas as pd

COUNTRIES = ["SG", "MY", "ID", "PH", "TH", "VN", "IN"]
LANG_BY_COUNTRY = {"SG": "en", "MY": "ms", "ID": "id", "PH": "tl", "TH": "th", "VN": "vi", "IN": "hi"}
PLAN_PRICE_SGD = {"Free": 0.0, "Basic": 8.90, "Premium": 12.90}

rng = np.random.default_rng(42)
plans = ["Free"] * 56 + ["Basic"] * 28 + ["Premium"] * 16
plans.remove("Premium"); plans.remove("Basic"); plans.remove("Free")
rng.shuffle(plans)

rows = [
    (1, "Ravi Kumar", "IN", "Premium", False),
    (2, "Siti Rahman", "MY", "Basic", False),
    (3, "Nguyen Van Minh", "VN", "Free", True),
]
for idx, plan in enumerate(plans, start=4):
    country = COUNTRIES[(idx * 3) % len(COUNTRIES)]
    _signup_offset = int(rng.integers(0, 500))  # drawn in app.py too -- keeps the RNG in sync
    churned = bool(rng.random() < 0.18)
    rows.append((idx, f"User{idx:03d}", country, plan, churned))

users_df = pd.DataFrame(rows, columns=["user_id", "name", "country", "plan", "churned"])

total_users = len(users_df)
churn_rate = users_df["churned"].mean() * 100
mrr = users_df.loc[~users_df["churned"], "plan"].map(PLAN_PRICE_SGD).sum()

print(f"Total users: {total_users}")
print(f"Churn rate: {churn_rate:.1f}%")
print(f"MRR (S$): {mrr:,.2f}")
print(users_df['plan'].value_counts())

```
Total users: 100
Churn rate: 24.0%
MRR (S$): 363.50
plan
Free       56
Basic      28
Premium    16
Name: count, dtype: int64
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
session_state = {}

def script_run(run_number, click_happened):
    # Plain variable -- recreated every rerun
    page_views = 0
    page_views += 1

    # session_state -- persists across reruns
    if "total_clicks" not in session_state:
        session_state["total_clicks"] = 0
    if click_happened:
        session_state["total_clicks"] += 1

    print(f"Run {run_number}: page_views={page_views}, "
          f"total_clicks={session_state['total_clicks']}")

script_run(1, click_happened=True)
script_run(2, click_happened=True)
script_run(3, click_happened=False)

```
Run 1: page_views=1, total_clicks=1
Run 2: page_views=1, total_clicks=2
Run 3: page_views=1, total_clicks=2
```

In [ ]:
premium_id_count = len(users_df[(users_df["country"] == "ID") & (users_df["plan"] == "Premium")])
print(premium_id_count)

In [ ]:
import streamlit as st

def get_data(refresh_token):
    # pretend this is an expensive load
    return {"rows": 1000, "loaded_with_token": refresh_token}

@st.cache_data
def cached_get_data(refresh_token):
    return get_data(refresh_token)

# In the main script:
import time
refresh_token = time.time()  # <-- new value EVERY rerun
data = cached_get_data(refresh_token)

In [ ]:
if "refresh_token" not in st.session_state:
    st.session_state.refresh_token = 0  # stable across reruns

if st.button("Refresh data"):
    st.session_state.refresh_token += 1  # only changes on explicit refresh

data = cached_get_data(st.session_state.refresh_token)

---

# Chapter 61: Backend Integration & Naive RAG

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install faiss-cpu numpy pandas

### 2.1 SQL backend integration with caching

In [ ]:
import sqlite3
import pandas as pd

# In a Streamlit app, this function would be decorated with @st.cache_resource
# so the connection is created once and reused across reruns.
def get_connection():
    conn = sqlite3.connect(":memory:")
    return conn

# In a Streamlit app, this function would be decorated with @st.cache_data
# so repeated calls with the same SQL string return a cached DataFrame
# instead of re-querying.
def run_query(conn, sql, params=()):
    return pd.read_sql(sql, conn, params=params)


# --- Demo: set up a tiny "users" table and query it ---
conn = get_connection()

users_df = pd.DataFrame([
    (1, "Ravi Kumar", "IN", "Premium", False),
    (2, "Siti Rahman", "MY", "Basic", False),
    (3, "Nguyen Van Minh", "VN", "Free", True),
    (4, "User004", "TH", "Basic", False),
    (5, "User005", "ID", "Premium", True),
], columns=["user_id", "name", "country", "plan", "churned"])
users_df.to_sql("users", conn, index=False, if_exists="replace")

result = run_query(conn, "SELECT country, COUNT(*) AS n FROM users WHERE plan != 'Free' GROUP BY country")
print(result)

```
  country  n
0      ID  1
1      IN  1
2      MY  1
3      TH  1
```

### 2.2 Why caching matters for backend calls specifically

In [ ]:
import time

# A simple memoization wrapper, standing in for @st.cache_data,
# now applied to something that simulates network latency.
def cache_data(func):
    cache = {}
    def wrapper(*args):
        if args in cache:
            return cache[args], "HIT"
        result = func(*args)
        cache[args] = result
        return result, "MISS"
    return wrapper


@cache_data
def query_country_counts(min_signup_year):
    time.sleep(0.01)  # simulate network round-trip
    # pretend this runs a real SQL query filtered by year
    return {"SG": 12, "MY": 18, "ID": 25, "PH": 14, "TH": 9, "VN": 11, "IN": 11}


# Simulate 3 reruns: page load, user clicks an unrelated checkbox, user changes the year filter
for label, year in [("page load", 2023), ("unrelated click", 2023), ("year filter changed", 2024)]:
    result, status = query_country_counts(year)
    print(f"{label}: cache {status}, result keys = {list(result.keys())[:3]}...")

```
page load: cache MISS, result keys = ['SG', 'MY', 'ID']...
unrelated click: cache HIT, result keys = ['SG', 'MY', 'ID']...
year filter changed: cache MISS, result keys = ['SG', 'MY', 'ID']...
```

### 2.3 Building a naive RAG pipeline from scratch

In [ ]:
import numpy as np

def build_vocab(texts):
    """Assign each unique word a position (index) in the vector."""
    vocab = {}
    for t in texts:
        for word in t.lower().replace(".", "").split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab


def embed(text, vocab):
    """Turn text into a normalized word-count vector."""
    vec = np.zeros(len(vocab), dtype="float32")
    for word in text.lower().replace(".", "").split():
        if word in vocab:
            vec[vocab[word]] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


texts = [
    "Premium users in Indonesia are watching less this month",
    "Churn rate increased for Basic plan subscribers in Vietnam",
    "Top genre by watch minutes is Thriller across all countries",
]

vocab = build_vocab(texts)
print(f"Vocabulary size: {len(vocab)} unique words")

vectors = [embed(t, vocab) for t in texts]
for t, v in zip(texts, vectors):
    print(f"  '{t[:40]}...' -> vector with {int((v != 0).sum())} non-zero entries")

```
Vocabulary size: 27 unique words
  'Premium users in Indonesia are watching ...' -> vector with 9 non-zero entries
  'Churn rate increased for Basic plan subs...' -> vector with 9 non-zero entries
  'Top genre by watch minutes is Thriller a...' -> vector with 10 non-zero entries
```

### 2.4 Storing vectors and searching with FAISS

In [ ]:
import faiss
import numpy as np

# Reuse the vocab/embed functions and texts from 2.3
doc_vectors = np.array([embed(t, vocab) for t in texts], dtype="float32")
print("doc_vectors shape:", doc_vectors.shape)  # (3 docs, 27 dims)

# IndexFlatIP = "flat" (brute-force) index using Inner Product.
# Because our vectors are normalized to length 1, inner product == cosine similarity.
index = faiss.IndexFlatIP(doc_vectors.shape[1])
index.add(doc_vectors)
print("Vectors in index:", index.ntotal)

# A new question, embedded the same way
query = "Why is churn going up for Basic users?"
query_vector = embed(query, vocab).reshape(1, -1)

scores, indices = index.search(query_vector, k=2)  # top 2 matches
print("\nTop matches for:", repr(query))
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"  #{rank}: score={score:.3f} -> {texts[idx]}")

```
doc_vectors shape: (3, 27)
Vectors in index: 3

Top matches for: 'Why is churn going up for Basic users?'
  #1: score=0.500 -> Churn rate increased for Basic plan subscribers in Vietnam
  #2: score=0.158 -> Top genre by watch minutes is Thriller across all countries
```

In [ ]:
print(f"k=2 requested, got {len(scores[0])} results, indices={indices[0].tolist()}")

```
k=2 requested, got 2 results, indices=[1, 2]
```

### 2.5 The full naive RAG loop: retrieve, then prompt

In [ ]:
def retrieve(query, vocab, index, docs, k=1):
    qvec = embed(query, vocab).reshape(1, -1)
    scores, idxs = index.search(qvec, k)
    return [(docs[i], float(s)) for i, s in zip(idxs[0], scores[0]) if i != -1]


def build_prompt(question, retrieved_docs):
    context = "\n".join(f"- {d['title']}: {d['text']}" for d, _ in retrieved_docs)
    return (
        "You are an internal data assistant.\n"
        "Use the context below to answer the question. "
        "If the context doesn't contain the answer, say so honestly.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )


# A tiny "knowledge base" of internal docs
docs = [
    {"id": "doc1", "title": "Churn Definition",
     "text": "A user is churned if their subscription end date is set and in the past."},
    {"id": "doc2", "title": "Plan Pricing",
     "text": "Basic plan costs S dollars 8.90 per month. Premium plan costs S dollars 12.90 per month."},
]
kb_vocab = build_vocab([d["text"] for d in docs])
kb_vectors = np.array([embed(d["text"], kb_vocab) for d in docs], dtype="float32")
kb_index = faiss.IndexFlatIP(kb_vectors.shape[1])
kb_index.add(kb_vectors)

question = "How much does the Premium plan cost?"
retrieved = retrieve(question, kb_vocab, kb_index, docs, k=1)
print("Retrieved doc:", retrieved[0][0]["title"], f"(score={retrieved[0][1]:.3f})")
print()
print(build_prompt(question, retrieved))

```
Retrieved doc: Plan Pricing (score=0.327)

You are an internal data assistant.
Use the context below to answer the question. If the context doesn't contain the answer, say so honestly.

Context:
- Plan Pricing: Basic plan costs S dollars 8.90 per month. Premium plan costs S dollars 12.90 per month.

Question: How much does the Premium plan cost?

Answer:
```

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import numpy as np
import faiss
import pandas as pd

# --- Backend: SQLite connection (stand-in for the warehouse read replica) ---
conn = sqlite3.connect(":memory:")

users_df = pd.DataFrame([
    (1, "Ravi Kumar", "ravi@example.com", "IN", "hi", "Premium", "2024-01-15", False),
    (2, "Siti Rahman", "siti@example.com", "MY", "ms", "Basic", "2024-02-03", False),
    (3, "Nguyen Van Minh", "minh@example.com", "VN", "vi", "Free", "2024-03-22", True),
    (4, "User004", "user004@example.com", "ID", "id", "Premium", "2023-11-02", False),
    (5, "User005", "user005@example.com", "ID", "id", "Basic", "2024-02-19", False),
    (6, "User006", "user006@example.com", "TH", "th", "Premium", "2023-08-30", True),
], columns=["user_id", "name", "email", "country", "language_pref", "plan", "signup_date", "churned"])
users_df.to_sql("users", conn, index=False, if_exists="replace")

# --- Knowledge base: CinemaStream facts (would live in cinemastream/docs/) ---
knowledge_base = [
    {"id": "kb1", "title": "Plan Pricing",
     "text": "Free plan costs zero. Basic plan costs 8.90 SGD per month. Premium plan costs 12.90 SGD per month."},
    {"id": "kb2", "title": "Churn Definition",
     "text": "A user has churned if the churned column is true, meaning their subscription has ended."},
    {"id": "kb3", "title": "Country Coverage",
     "text": "CinemaStream operates in seven countries: Singapore Malaysia Indonesia Philippines Thailand Vietnam and India."},
]

def build_vocab(texts):
    vocab = {}
    for t in texts:
        for w in t.lower().replace(".", "").replace(",", "").split():
            if w not in vocab:
                vocab[w] = len(vocab)
    return vocab

def embed(text, vocab):
    vec = np.zeros(len(vocab), dtype="float32")
    for w in text.lower().replace(".", "").replace(",", "").split():
        if w in vocab:
            vec[vocab[w]] += 1.0
    n = np.linalg.norm(vec)
    return vec / n if n > 0 else vec

kb_vocab = build_vocab([d["text"] for d in knowledge_base])
kb_vectors = np.array([embed(d["text"], kb_vocab) for d in knowledge_base], dtype="float32")
kb_index = faiss.IndexFlatIP(kb_vectors.shape[1])
kb_index.add(kb_vectors)


def ask_anything(question, fallback_threshold=0.25):
    """Tool-routing + RAG fallback:
    1. If the question matches a known LIVE-DATA pattern, run a real SQL
       query and ground the prompt in that result (no KB retrieval needed).
    2. Otherwise, retrieve from the static knowledge base. If nothing is
       relevant enough (score < fallback_threshold), return Wei Lin's
       fallback message instead of an empty/confusing prompt."""
    q_lower = question.lower()

    # Live-data routing: a question about Premium users *by country* needs
    # a real query, not a static fact -- regardless of what the KB retrieval
    # would say.
    if "premium" in q_lower and "countr" in q_lower:
        result = pd.read_sql(
            "SELECT country, COUNT(*) AS premium_users FROM users "
            "WHERE plan = 'Premium' AND churned = 0 GROUP BY country ORDER BY premium_users DESC",
            conn,
        )
        context = "Live query result (active Premium users by country):\n" + result.to_string(index=False)
        return (
            "You are CinemaStream's internal data assistant.\n"
            "Use the context below to answer the question.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\nAnswer:"
        )

    # Static knowledge-base retrieval
    qvec = embed(question, kb_vocab).reshape(1, -1)
    scores, idxs = kb_index.search(qvec, 1)
    best_score = float(scores[0][0])
    best_idx = int(idxs[0][0])

    if best_score < fallback_threshold:
        return "I couldn't find anything relevant to that question in the CinemaStream knowledge base yet."

    matched_doc = knowledge_base[best_idx]
    context = f"- {matched_doc['title']}: {matched_doc['text']}"
    return (
        "You are CinemaStream's internal data assistant.\n"
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer:"
    )


# Rohan's actual question from the Slack thread -- live-data routing fires
print(ask_anything("How many Premium users do we have in each country?"))
print("\n" + "=" * 60 + "\n")
# A question with no relevant context (Wei Lin's fallback case)
print(ask_anything("What's the weather like in Manila today?"))

```
You are CinemaStream's internal data assistant.
Use the context below to answer the question.

Context:
Live query result (active Premium users by country):
country  premium_users
     IN              1
     ID              1

Question: How many Premium users do we have in each country?

Answer:

============================================================

I couldn't find anything relevant to that question in the CinemaStream knowledge base yet.
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
print(ask_anything("What does the Basic plan cost?"))

```
You are CinemaStream's internal data assistant.
Use the context below to answer the question.

Context:
- Plan Pricing: Free plan costs zero. Basic plan costs 8.90 SGD per month. Premium plan costs 12.90 SGD per month.

Question: What does the Basic plan cost?

Answer:
```

---

# Chapter 62: Deployment

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Streamlit Community Cloud — the zero-infrastructure path

In [ ]:
# What Streamlit Community Cloud needs to find in your repo
deployment_requirements = {
    "entry_point": "cinemastream/streamlit_app/app.py",
    "dependencies_file": "requirements.txt (at repo root, or specify path in app settings)",
    "python_version": "specified via runtime.txt or pyproject.toml (optional, defaults to a recent 3.x)",
    "secrets": "set via the Streamlit Cloud dashboard -> Settings -> Secrets (TOML format), "
               "NEVER committed to the repo",
}

for key, value in deployment_requirements.items():
    print(f"{key}: {value}")

```
entry_point: cinemastream/streamlit_app/app.py
dependencies_file: requirements.txt (at repo root, or specify path in app settings)
python_version: specified via runtime.txt or pyproject.toml (optional, defaults to a recent 3.x)
secrets: set via the Streamlit Cloud dashboard -> Settings -> Secrets (TOML format), NEVER committed to the repo
```

### 2.2 Writing `requirements.txt` for the Streamlit app

In [ ]:
# Validate a requirements.txt: every pinned package should correspond to
# something the app actually imports. This is the kind of check an FDE
# runs before a deploy -- "why is this 400MB image only running a
# 200-line dashboard?"

app_imports = {"streamlit", "pandas", "numpy"}  # from app.py's import statements (Ch 60)
rag_imports = {"faiss-cpu"}  # from Ch 61's RAG demo, if wired into app.py

requirements_txt = app_imports | rag_imports

print("requirements.txt would contain:")
for pkg in sorted(requirements_txt):
    print(f"  {pkg}")

# Sanity check: nothing in requirements.txt that app.py doesn't import
unused = requirements_txt - (app_imports | rag_imports)
print(f"\nUnused entries: {unused if unused else 'none'}")

```
requirements.txt would contain:
  faiss-cpu
  numpy
  pandas
  streamlit

Unused entries: none
```

### 2.3 Hugging Face Spaces

In [ ]:
# The YAML frontmatter Hugging Face Spaces reads from README.md
# to know how to build and run the app.
space_config = """\
---
title: CinemaStream Analytics
emoji: 📊
colorFrom: blue
colorTo: green
sdk: streamlit
sdk_version: "1.30.0"
app_file: app.py
pinned: false
---
"""

print(space_config)

# Validate: required keys present?
import re
required_keys = {"title", "sdk", "app_file"}
found_keys = set(re.findall(r"^(\w+):", space_config, re.MULTILINE))
missing = required_keys - found_keys
print(f"Missing required keys: {missing if missing else 'none'}")

```
---
title: CinemaStream Analytics
emoji: 📊
colorFrom: blue
colorTo: green
sdk: streamlit
sdk_version: "1.30.0"
app_file: app.py
pinned: false
---

Missing required keys: none
```

### 2.4 Docker + Cloud Run — the full-control path

In [ ]:
# The Dockerfile content for the Streamlit app
dockerfile_content = """\
FROM python:3.11-slim

WORKDIR /app

COPY cinemastream/streamlit_app/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY cinemastream/streamlit_app/app.py .

EXPOSE 8501

HEALTHCHECK CMD curl --fail http://localhost:8501/_stcore/health || exit 1

ENTRYPOINT ["streamlit", "run", "app.py", \\
            "--server.port=8501", \\
            "--server.address=0.0.0.0", \\
            "--server.headless=true"]
"""

print(dockerfile_content)

# Basic structural validation -- the kind of check a pre-commit hook might run
required_instructions = ["FROM", "WORKDIR", "COPY", "RUN", "EXPOSE", "ENTRYPOINT"]
present = [instr for instr in required_instructions if instr in dockerfile_content]
print(f"Instructions present: {present}")
print(f"All required instructions present: {len(present) == len(required_instructions)}")

```
FROM python:3.11-slim

WORKDIR /app

COPY cinemastream/streamlit_app/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY cinemastream/streamlit_app/app.py .

EXPOSE 8501

HEALTHCHECK CMD curl --fail http://localhost:8501/_stcore/health || exit 1

ENTRYPOINT ["streamlit", "run", "app.py", \
            "--server.port=8501", \
            "--server.address=0.0.0.0", \
            "--server.headless=true"]

Instructions present: ['FROM', 'WORKDIR', 'COPY', 'RUN', 'EXPOSE', 'ENTRYPOINT']
All required instructions present: True
```

In [ ]:
# The command sequence for Docker + Cloud Run deployment
commands = [
    "docker build -t cinemastream-dashboard:latest -f cinemastream/streamlit_app/Dockerfile .",
    "docker run -p 8501:8501 cinemastream-dashboard:latest   # local test first",
    "docker tag cinemastream-dashboard:latest gcr.io/PROJECT_ID/cinemastream-dashboard",
    "docker push gcr.io/PROJECT_ID/cinemastream-dashboard",
    "gcloud run deploy cinemastream-dashboard "
    "--image gcr.io/PROJECT_ID/cinemastream-dashboard "
    "--platform managed --region asia-southeast1 --allow-unauthenticated",
]

for i, cmd in enumerate(commands, start=1):
    print(f"{i}. {cmd}")

```
1. docker build -t cinemastream-dashboard:latest -f cinemastream/streamlit_app/Dockerfile .
2. docker run -p 8501:8501 cinemastream-dashboard:latest   # local test first
3. docker tag cinemastream-dashboard:latest gcr.io/PROJECT_ID/cinemastream-dashboard
4. docker push gcr.io/PROJECT_ID/cinemastream-dashboard
5. gcloud run deploy cinemastream-dashboard --image gcr.io/PROJECT_ID/cinemastream-dashboard --platform managed --region asia-southeast1 --allow-unauthenticated
```

### 2.5 Comparing the three paths

In [ ]:
comparison = {
    "Streamlit Community Cloud": {
        "setup_time": "~5 minutes",
        "cost": "free (public repos), usage limits apply",
        "control": "low -- platform-managed Python env",
        "best_for": "internal prototypes, public portfolio demos",
    },
    "Hugging Face Spaces": {
        "setup_time": "~10 minutes",
        "cost": "free tier available, paid for more CPU/GPU",
        "control": "low-medium -- Docker option available for more control",
        "best_for": "tools that also use HF models/datasets",
    },
    "Docker + Cloud Run": {
        "setup_time": "~1-2 hours first time, minutes after",
        "cost": "pay-per-use, scales to zero when idle",
        "control": "high -- full container control, fits existing infra",
        "best_for": "production internal tools, company-standard infra",
    },
}

for platform, attrs in comparison.items():
    print(f"\n{platform}:")
    for k, v in attrs.items():
        print(f"  {k}: {v}")

```

Streamlit Community Cloud:
  setup_time: ~5 minutes
  cost: free (public repos), usage limits apply
  control: low -- platform-managed Python env
  best_for: internal prototypes, public portfolio demos

Hugging Face Spaces:
  setup_time: ~10 minutes
  cost: free tier available, paid for more CPU/GPU
  control: low-medium -- Docker option available for more control
  best_for: tools that also use HF models/datasets

Docker + Cloud Run:
  setup_time: ~1-2 hours first time, minutes after
  cost: pay-per-use, scales to zero when idle
  control: high -- full container control, fits existing infra
  best_for: production internal tools, company-standard infra
```

## 3. CinemaStream in Practice

In [ ]:
# cinemastream/streamlit_app/requirements.txt content
requirements_content = """\
streamlit>=1.30.0
pandas>=2.2.0
numpy>=1.26.0
faiss-cpu>=1.7.4
"""
print(requirements_content)

```
streamlit>=1.30.0
pandas>=2.2.0
numpy>=1.26.0
faiss-cpu>=1.7.4
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

```
ENTRYPOINT ["streamlit", "run", "app.py", "--server.port=8501"]
```

---

# Chapter 63: User-Centered Design

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Personas as structured data

In [ ]:
# A persona represented as a dictionary -- structured enough to check
# design decisions against, narrative enough to keep "a real person" in mind.

persona_alex = {
    "name": "Alex",
    "role": "Regional Manager",
    "technical_comfort": "low",       # low / medium / high
    "primary_device": "mobile",       # mobile / desktop / tablet
    "time_per_session_min": 2,        # how long they'll realistically look
    "top_goal": "see this week's numbers vs last week, at a glance",
    "frustration": "dashboards that require scrolling sideways on a phone",
}

persona_jordan = {
    "name": "Jordan",
    "role": "Data Analyst",
    "technical_comfort": "high",
    "primary_device": "desktop",
    "time_per_session_min": 45,
    "top_goal": "filter, drill down, and export data for deeper analysis",
    "frustration": "dashboards that hide the underlying numbers behind charts only",
}

for persona in (persona_alex, persona_jordan):
    print(f"{persona['name']} ({persona['role']}): "
          f"{persona['time_per_session_min']} min on {persona['primary_device']}, "
          f"wants: {persona['top_goal']}")

```
Alex (Regional Manager): 2 min on mobile, wants: see this week's numbers vs last week, at a glance
Jordan (Data Analyst): 45 min on desktop, wants: filter, drill down, and export data for deeper analysis
```

### 2.2 Checking a design decision against a persona

In [ ]:
def evaluate_design_for_persona(design, persona):
    """Return a list of concerns this design choice raises for this persona."""
    concerns = []

    if persona["primary_device"] == "mobile" and design.get("requires_horizontal_scroll"):
        concerns.append("Requires horizontal scroll -- bad on mobile")

    if persona["time_per_session_min"] <= 2 and design.get("num_clicks_to_key_metric", 0) > 1:
        concerns.append(
            f"Key metric needs {design['num_clicks_to_key_metric']} clicks "
            f"-- too many for a {persona['time_per_session_min']}-minute session"
        )

    if persona["technical_comfort"] == "low" and design.get("has_raw_sql_filter_box"):
        concerns.append("Raw SQL filter box -- not appropriate for low technical comfort")

    if persona["technical_comfort"] == "high" and design.get("hides_underlying_numbers"):
        concerns.append("Hides underlying numbers behind charts only -- analyst wants to drill down")

    return concerns if concerns else ["No concerns"]


# A design draft for a single dashboard screen
draft_design = {
    "requires_horizontal_scroll": True,
    "num_clicks_to_key_metric": 3,
    "has_raw_sql_filter_box": True,
    "hides_underlying_numbers": True,
}

for persona in (persona_alex, persona_jordan):
    print(f"\n{persona['name']}:")
    for concern in evaluate_design_for_persona(draft_design, persona):
        print(f"  - {concern}")

```

Alex:
  - Requires horizontal scroll -- bad on mobile
  - Key metric needs 3 clicks -- too many for a 2-minute session
  - Raw SQL filter box -- not appropriate for low technical comfort

Jordan:
  - Hides underlying numbers behind charts only -- analyst wants to drill down
```

### 2.3 Wireframes as structured layout data

In [ ]:
# A wireframe for a mobile-first summary screen, represented as regions.
# 'priority' (1 = highest) roughly maps to vertical position and size.

mobile_wireframe = [
    {"region": "header", "content": "Page title + date range", "priority": 3},
    {"region": "key_metric", "content": "This week's MRR, large text, vs last week delta", "priority": 1},
    {"region": "secondary_metrics", "content": "Active users, churn rate -- smaller cards", "priority": 2},
    {"region": "chart", "content": "7-day trend line, single metric", "priority": 2},
    {"region": "footer", "content": "Link: 'View full dashboard on desktop'", "priority": 4},
]

# Sort by priority to simulate "what appears first when scrolling"
for region in sorted(mobile_wireframe, key=lambda r: r["priority"]):
    print(f"[{region['priority']}] {region['region']}: {region['content']}")

```
[1] key_metric: This week's MRR, large text, vs last week delta
[2] secondary_metrics: Active users, churn rate -- smaller cards
[2] chart: 7-day trend line, single metric
[3] header: Page title + date range
[4] footer: Link: 'View full dashboard on desktop'
```

### 2.4 Role-based access control (RBAC)

In [ ]:
# Roles: each role is a set of permission strings
roles = {
    "viewer": {"view_summary", "view_charts"},
    "analyst": {"view_summary", "view_charts", "view_raw_data", "export_csv"},
    "finance": {"view_summary", "view_charts", "view_raw_data", "view_revenue_breakdown", "export_csv"},
    "admin": {"view_summary", "view_charts", "view_raw_data", "view_revenue_breakdown",
              "export_csv", "manage_users"},
}

# Users: each user has one or more roles
users = {
    "rohan": {"analyst"},
    "dharani": {"finance"},
    "priya": {"admin"},
    "wei_lin": {"viewer"},
}


def effective_permissions(user_id):
    """Union of all permissions across a user's roles."""
    perms = set()
    for role in users[user_id]:
        perms |= roles[role]
    return perms


def can(user_id, permission):
    return permission in effective_permissions(user_id)


for user_id in users:
    perms = sorted(effective_permissions(user_id))
    print(f"{user_id}: {perms}")

print()
print("Can Rohan see revenue breakdown?", can("rohan", "view_revenue_breakdown"))
print("Can Dharani see revenue breakdown?", can("dharani", "view_revenue_breakdown"))
print("Can Wei Lin export CSV?", can("wei_lin", "export_csv"))

```
rohan: ['export_csv', 'view_charts', 'view_raw_data', 'view_summary']
dharani: ['export_csv', 'view_charts', 'view_raw_data', 'view_revenue_breakdown', 'view_summary']
priya: ['export_csv', 'manage_users', 'view_charts', 'view_raw_data', 'view_revenue_breakdown', 'view_summary']
wei_lin: ['view_charts', 'view_summary', 'view_summary']
```

In [ ]:
# Re-running the same check, output transcribed faithfully this time
for user_id in users:
    perms = sorted(effective_permissions(user_id))
    print(f"{user_id}: {perms}")

```
rohan: ['export_csv', 'view_charts', 'view_raw_data', 'view_summary']
dharani: ['export_csv', 'view_charts', 'view_raw_data', 'view_revenue_breakdown', 'view_summary']
priya: ['export_csv', 'manage_users', 'view_charts', 'view_raw_data', 'view_revenue_breakdown', 'view_summary']
wei_lin: ['view_charts', 'view_summary']
```

### 2.5 Accessibility checks on a design

In [ ]:
def relative_luminance(rgb):
    """Approximate relative luminance of an RGB color (0-255 each channel)."""
    r, g, b = [c / 255 for c in rgb]
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def contrast_ratio(rgb1, rgb2):
    """WCAG-style contrast ratio between two colors. >= 4.5 passes for normal text."""
    l1, l2 = relative_luminance(rgb1), relative_luminance(rgb2)
    lighter, darker = max(l1, l2), min(l1, l2)
    return (lighter + 0.05) / (darker + 0.05)


# Candidate color pairs (text_color, background_color)
candidates = {
    "teal text on cream":   ((61, 115, 112), (245, 239, 230)),
    "light grey on white":  ((220, 220, 220), (255, 255, 255)),
    "orange text on cream": ((224, 122, 59), (245, 239, 230)),
}

for label, (text_rgb, bg_rgb) in candidates.items():
    ratio = contrast_ratio(text_rgb, bg_rgb)
    passes = ratio >= 4.5
    print(f"{label}: ratio={ratio:.2f}, passes WCAG AA (>=4.5)? {passes}")

```
teal text on cream: ratio=2.17, passes WCAG AA (>=4.5)? False
light grey on white: ratio=1.15, passes WCAG AA (>=4.5)? False
orange text on cream: ratio=1.66, passes WCAG AA (>=4.5)? False
```

In [ ]:
# A status indicator design -- does it rely on color alone?
status_indicators = [
    {"status": "healthy", "color": "green", "icon": None,  "label_text": None},
    {"status": "warning", "color": "orange", "icon": "!",  "label_text": "Warning"},
    {"status": "critical", "color": "red", "icon": "X",    "label_text": "Critical"},
]

for indicator in status_indicators:
    color_only = indicator["icon"] is None and indicator["label_text"] is None
    print(f"{indicator['status']}: color={indicator['color']}, "
          f"conveyed by color alone? {color_only}")

```
healthy: color=green, conveyed by color alone? True
warning: color=orange, conveyed by color alone? False
critical: color=red, conveyed by color alone? False
```

## 3. CinemaStream in Practice

In [ ]:
# CinemaStream dashboard personas, built from the cast you already know
persona_priya = {
    "name": "Priya",
    "role": "Head of Analytics",
    "technical_comfort": "high",
    "primary_device": "mobile",   # checks dashboard between meetings, per Ch 62
    "time_per_session_min": 2,
    "top_goal": "spot anomalies in MRR or churn before a meeting",
    "frustration": "having to scroll sideways or wait for a chart to load on 4G",
}

persona_rohan = {
    "name": "Rohan",
    "role": "Product Manager",
    "technical_comfort": "medium",
    "primary_device": "desktop",
    "time_per_session_min": 5,
    "top_goal": "get a specific number for a Slack reply -- 'can I get a number by EOD?'",
    "frustration": "dashboards where the number he needs is three clicks behind a filter",
}

persona_dharani = {
    "name": "Dharani",
    "role": "CFO",
    "technical_comfort": "medium",
    "primary_device": "desktop",
    "time_per_session_min": 10,
    "top_goal": "see revenue breakdown by plan and country, in SGD",
    "frustration": "numbers shown in local currency without an SGD total",
}

for p in (persona_priya, persona_rohan, persona_dharani):
    print(f"{p['name']} ({p['role']}): {p['primary_device']}, "
          f"{p['time_per_session_min']} min -- {p['top_goal']}")

```
Priya (Head of Analytics): mobile, 2 min -- spot anomalies in MRR or churn before a meeting
Rohan (Product Manager): desktop, 5 min -- get a specific number for a Slack reply -- 'can I get a number by EOD?'
Dharani (CFO): desktop, 10 min -- see revenue breakdown by plan and country, in SGD
```

In [ ]:
cinemastream_mobile_wireframe = [
    {"region": "key_metric", "content": "MRR this week: S$341.70 (+2.1% vs last week)", "priority": 1},
    {"region": "alert_banner", "content": "Only shown if churn rate > threshold -- e.g., 'Churn up in VN'", "priority": 1},
    {"region": "secondary_metrics", "content": "Active users by plan -- 3 small cards", "priority": 2},
    {"region": "trend_chart", "content": "7-day MRR trend, single line, no legend needed", "priority": 2},
    {"region": "link_to_desktop", "content": "'View full breakdown (best on desktop)'", "priority": 3},
]

for region in sorted(cinemastream_mobile_wireframe, key=lambda r: r["priority"]):
    print(f"[{region['priority']}] {region['region']}: {region['content']}")

```
[1] key_metric: MRR this week: S$341.70 (+2.1% vs last week)
[1] alert_banner: Only shown if churn rate > threshold -- e.g., 'Churn up in VN'
[2] secondary_metrics: Active users by plan -- 3 small cards
[2] trend_chart: 7-day MRR trend, single line, no legend needed
[3] link_to_desktop: 'View full breakdown (best on desktop)'
```

In [ ]:
cinemastream_roles = {
    "viewer": {"view_mrr_summary", "view_active_users", "view_top_genres"},
    "pm": {"view_mrr_summary", "view_active_users", "view_top_genres", "view_churn_by_country"},
    "finance": {"view_mrr_summary", "view_active_users", "view_top_genres",
                "view_churn_by_country", "view_revenue_breakdown_sgd"},
    "admin": {"view_mrr_summary", "view_active_users", "view_top_genres",
              "view_churn_by_country", "view_revenue_breakdown_sgd", "manage_dashboard_users"},
}

cinemastream_users = {
    "rohan": {"pm"},
    "dharani": {"finance"},
    "priya": {"admin"},
}


def cs_effective_permissions(user_id):
    perms = set()
    for role in cinemastream_users[user_id]:
        perms |= cinemastream_roles[role]
    return perms


for user_id in cinemastream_users:
    print(f"{user_id}: {sorted(cs_effective_permissions(user_id))}")

```
rohan: ['view_active_users', 'view_churn_by_country', 'view_mrr_summary', 'view_top_genres']
dharani: ['view_active_users', 'view_churn_by_country', 'view_mrr_summary', 'view_revenue_breakdown_sgd', 'view_top_genres']
priya: ['view_revenue_breakdown_sgd', 'view_active_users', 'view_churn_by_country', 'view_mrr_summary', 'view_top_genres', 'manage_dashboard_users', 'manage_active_users']
```

```
rohan: ['view_active_users', 'view_churn_by_country', 'view_mrr_summary', 'view_top_genres']
dharani: ['view_active_users', 'view_churn_by_country', 'view_mrr_summary', 'view_revenue_breakdown_sgd', 'view_top_genres']
priya: ['manage_dashboard_users', 'view_active_users', 'view_churn_by_country', 'view_mrr_summary', 'view_revenue_breakdown_sgd', 'view_top_genres']
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
roles = {
    "viewer": {"read"},
    "editor": {"read", "write"},
    "owner": {"read", "write", "delete"},
}

users = {
    "sam": {"viewer", "editor"},
}

In [ ]:
def effective_permissions(user_id):
    perms = set()
    for role in users[user_id]:
        perms |= roles[role]
    return perms

print(effective_permissions("sam"))

```
{'read', 'write'}
```

In [ ]:
persona_carlos = {
    "name": "Carlos",
    "role": "Backend Engineer",
    "technical_comfort": "high",
    "primary_device": "desktop",
    "time_per_session_min": 3,
    "top_goal": "verify a just-deployed pipeline fix produced sane row counts and recent timestamps",
    "frustration": "summary-only views that hide raw counts and last-updated timestamps",
}
print(persona_carlos)

```
{'name': 'Carlos', 'role': 'Backend Engineer', 'technical_comfort': 'high', 'primary_device': 'desktop', 'time_per_session_min': 3, 'top_goal': 'verify a just-deployed pipeline fix produced sane row counts and recent timestamps', 'frustration': 'summary-only views that hide raw counts and last-updated timestamps'}
```

---

# Chapter 64: Prompt Engineering Fundamentals

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The four-part prompt structure

In [ ]:
# A prompt represented as its four components, then assembled into one string.
# This is the "specification" view of a prompt -- each part has a job.

prompt_parts = {
    "role": "You are a careful assistant that answers questions about a small inventory database.",
    "task": "Given the inventory data below, answer the user's question.",
    "context": "Inventory: apples=12, bananas=0, cherries=45, dates=3",
    "format": "Respond with a single sentence. If an item has 0 stock, say 'out of stock'.",
    "user_query": "How many bananas do we have?",
}

assembled_prompt = (
    f"{prompt_parts['role']}\n\n"
    f"Task: {prompt_parts['task']}\n"
    f"Context: {prompt_parts['context']}\n"
    f"Format: {prompt_parts['format']}\n\n"
    f"Question: {prompt_parts['user_query']}"
)

print(assembled_prompt)

```
You are a careful assistant that answers questions about a small inventory database.

Task: Given the inventory data below, answer the user's question.
Context: Inventory: apples=12, bananas=0, cherries=45, dates=3
Format: Respond with a single sentence. If an item has 0 stock, say 'out of stock'.

Question: How many bananas do we have?
```

In [ ]:
# A simulated model: a tiny rule-based stand-in for an LLM, used to
# demonstrate how prompt STRUCTURE affects what kind of output you get.
# Real LLMs are far more flexible -- this mock only exists to make
# the effect of missing components visible and testable.

def simulated_model(prompt_parts):
    """Returns a plausible response based on which parts are present."""
    if "context" not in prompt_parts:
        return "I don't have access to the inventory data, but typically bananas are a common stock item."
    if "format" not in prompt_parts:
        return "Looking at the inventory data, bananas: 0. Apples: 12. Cherries: 45. Dates: 3."
    # All parts present: focused, formatted answer
    inventory = dict(item.split("=") for item in prompt_parts["context"].split(": ")[1].split(", "))
    item = "bananas"
    count = int(inventory[item])
    if count == 0:
        return f"We are currently out of stock on {item}."
    return f"We have {count} {item} in stock."


full_prompt = dict(prompt_parts)
no_context_prompt = {k: v for k, v in prompt_parts.items() if k != "context"}
no_format_prompt = {k: v for k, v in prompt_parts.items() if k != "format"}

print("Full prompt:    ", simulated_model(full_prompt))
print("No context:     ", simulated_model(no_context_prompt))
print("No format spec: ", simulated_model(no_format_prompt))

```
Full prompt:     We are currently out of stock on bananas.
No context:      I don't have access to the inventory data, but typically bananas are a common stock item.
No format spec:  Looking at the inventory data, bananas: 0. Apples: 12. Cherries: 45. Dates: 3.
```

### 2.2 Chain-of-thought prompting

In [ ]:
# Without chain-of-thought: ask for the final answer directly.
# With chain-of-thought: ask the model to show intermediate steps first.

direct_prompt = "A theater has 8 rows of 12 seats. 3 rows are reserved for staff. How many seats are available for the public?"

cot_prompt = (
    "A theater has 8 rows of 12 seats. 3 rows are reserved for staff. "
    "How many seats are available for the public? "
    "Think step by step before giving your final answer."
)


def simulated_model_direct(prompt):
    # A direct prompt sometimes skips a step under time/token pressure --
    # here it computes total seats but forgets to subtract the reserved rows.
    return "96"  # 8 * 12 = 96 -- WRONG, didn't subtract reserved rows


def simulated_model_cot(prompt):
    # With CoT, the intermediate steps are shown, making the subtraction explicit
    return (
        "Step 1: Total seats = 8 rows * 12 seats = 96.\n"
        "Step 2: Reserved rows = 3 rows * 12 seats = 36.\n"
        "Step 3: Available seats = 96 - 36 = 60.\n"
        "Final answer: 60"
    )


print("Direct prompt result:")
print(simulated_model_direct(direct_prompt))
print("\nChain-of-thought prompt result:")
print(simulated_model_cot(cot_prompt))

```
Direct prompt result:
96

Direct prompt result:
96
```

```
Direct prompt result:
96

Chain-of-thought prompt result:
Step 1: Total seats = 8 rows * 12 seats = 96.
Step 2: Reserved rows = 3 rows * 12 seats = 36.
Step 3: Available seats = 96 - 36 = 60.
Final answer: 60
```

### 2.3 Few-shot prompting

In [ ]:
# Zero-shot: describe the desired format in words
zero_shot_prompt = (
    "Classify the sentiment of this review as positive, negative, or mixed. "
    "Respond with only the single word.\n\n"
    "Review: 'The picture quality was great but it kept buffering every five minutes.'"
)

# Few-shot: show examples of the input -> output pattern, then the real input
few_shot_prompt = """\
Review: 'Loved every minute, will watch again.'
Sentiment: positive

Review: 'Completely unwatchable, kept crashing.'
Sentiment: negative

Review: 'The picture quality was great but it kept buffering every five minutes.'
Sentiment:"""

print("Zero-shot prompt:")
print(zero_shot_prompt)
print("\nFew-shot prompt:")
print(few_shot_prompt)

```
Zero-shot prompt:
Classify the sentiment of this review as positive, negative, or mixed. Respond with only the single word.

Review: 'The picture quality was great but it kept buffering every five minutes.'

Few-shot prompt:
Review: 'Loved every minute, will watch again.'
Sentiment: positive

Review: 'Completely unwatchable, kept crashing.'
Sentiment: negative

Review: 'The picture quality was great but it kept buffering every five minutes.'
Sentiment:
```

### 2.4 The 6-gap framework as a diagnostic function

In [ ]:
def diagnose_gap(symptoms):
    """
    Given a dict of observed symptoms about a failed LLM response,
    return the most likely Gap (1-6) and the category of fix.

    symptoms keys (booleans):
    - model_lacks_domain_fact: the model doesn't know a fact that exists
      somewhere (e.g., a recent CinemaStream policy change)
    - relevant_info_was_in_context_but_ignored: the answer was in the
      prompt, but the model didn't use it correctly
    - requires_inferring_cause_and_effect: the question asks "why did X
      happen" or "what would happen if we changed Y"
    - prompt_is_vague_or_ambiguous: the instructions could be read multiple ways
    - same_prompt_gave_different_answers_across_runs: re-running with
      identical input produced different outputs
    - output_format_is_unparseable: the response doesn't match the
      requested structure (e.g., invalid JSON)
    """
    if symptoms.get("output_format_is_unparseable"):
        return (6, "Structure", "Output parsing / structured output constraints (e.g., JSON schema, regex validation)")
    if symptoms.get("same_prompt_gave_different_answers_across_runs"):
        return (5, "Consistency", "Eval + SBQ measurement -- run N times, measure stability before trusting any single answer")
    if symptoms.get("prompt_is_vague_or_ambiguous"):
        return (4, "Strategy", "Prompt redesign -- clarify role/task/format")
    if symptoms.get("requires_inferring_cause_and_effect"):
        return (3, "Causal", "Fix lives outside the model -- needs a causal analysis pipeline, not a smarter prompt")
    if symptoms.get("relevant_info_was_in_context_but_ignored"):
        return (2, "Context", "Context engineering -- restructure what's in the prompt and where")
    if symptoms.get("model_lacks_domain_fact"):
        return (1, "Knowledge", "RAG -- retrieve the missing fact and add it to context")
    return (0, "Unknown", "Insufficient symptoms to diagnose -- gather more information")


# Three example failures
case_a = {"model_lacks_domain_fact": True}
case_b = {"relevant_info_was_in_context_but_ignored": True, "model_lacks_domain_fact": False}
case_c = {"output_format_is_unparseable": True, "relevant_info_was_in_context_but_ignored": True}

for name, case in [("Case A", case_a), ("Case B", case_b), ("Case C", case_c)]:
    gap_num, gap_name, fix = diagnose_gap(case)
    print(f"{name}: Gap {gap_num} ({gap_name}) -> {fix}")

```
Case A: Gap 1 (Knowledge) -> RAG -- retrieve the missing fact and add it to context
Case B: Gap 2 (Context) -> Context engineering -- restructure what's in the prompt and where
Case C: Gap 6 (Structure) -> Output parsing / structured output constraints (e.g., JSON schema, regex validation)
```

### 2.5 The intervention hierarchy in practice

In [ ]:
# Modeling the Intervention Hierarchy's claimed effect sizes as a simple
# "accuracy improvement" simulation. This is illustrative, not a real
# benchmark -- the point is the RELATIVE size of each lever's effect.

baseline_accuracy = 0.55  # 55% -- a locked failure mode, lots of wrong answers

interventions = {
    "Rank 3: bigger model only":       0.05,   # +5pp -- knowledge improves, reasoning gaps remain
    "Rank 2: better prompt only":      0.10,   # +10pp -- side effects possible, no consistency gain
    "Rank 1: context engineering":     0.40,   # +40pp -- the dominant lever
    "Rank 1 + Rank 2 combined":        0.45,   # context engineering plus a cleaned-up prompt
}

print(f"Baseline accuracy: {baseline_accuracy:.0%}\n")
for label, improvement in interventions.items():
    new_accuracy = min(baseline_accuracy + improvement, 1.0)
    print(f"{label}: {baseline_accuracy:.0%} -> {new_accuracy:.0%} (+{improvement:.0%})")

```
Baseline accuracy: 55%

Rank 3: bigger model only: 55% -> 60% (+5%)
Rank 2: better prompt only: 55% -> 65% (+10%)
Rank 1: context engineering: 55% -> 95% (+40%)
Rank 1 + Rank 2 combined: 55% -> 65% (+45%)
```

```
Baseline accuracy: 55%

Rank 3: bigger model only: 55% -> 60% (+5%)
Rank 2: better prompt only: 55% -> 65% (+10%)
Rank 1: context engineering: 55% -> 95% (+40%)
Rank 1 + Rank 2 combined: 55% -> 100% (+45%)
```

## 3. CinemaStream in Practice

In [ ]:
# Beta log entries: (question, tool's answer, what's actually wrong)
beta_failures = [
    {
        "question": "What's our refund policy for users in Vietnam?",
        "answer": "CinemaStream generally follows a 14-day refund policy across all regions.",
        "actual_issue": "CinemaStream has NO published refund policy document in the warehouse "
                         "or knowledge base -- the model is generating a plausible-sounding "
                         "generic SaaS answer, not retrieving a real CinemaStream policy.",
    },
    {
        "question": "Which country had the highest churn rate last month, and how does that compare to Premium users specifically?",
        "answer": "Vietnam had the highest churn rate.",
        "actual_issue": "The retrieved context DID include a per-plan churn breakdown table, "
                         "but the model's answer only addressed the first half of the question "
                         "(highest churn country) and ignored the Premium-specific comparison "
                         "that was sitting right there in the retrieved rows.",
    },
    {
        "question": "Give me this week's top 3 movies by completion rate as a JSON list.",
        "answer": "Sure! The top 3 movies by completion rate this week are:\n1. Monsoon Heart (89%)\n2. Hujan di Singapura (84%)\n3. Office Hari Ini (78%)",
        "actual_issue": "Rohan's downstream Slack-bot integration expects valid JSON to parse "
                         "and post as a formatted message. This response is human-readable but "
                         "not valid JSON -- the integration throws a parsing error.",
    },
]

for i, case in enumerate(beta_failures, start=1):
    print(f"Beta failure {i}:")
    print(f"  Q: {case['question']}")
    print(f"  A: {case['answer'][:80]}{'...' if len(case['answer']) > 80 else ''}")
    print(f"  Issue: {case['actual_issue']}")
    print()

```
Beta failure 1:
  Q: What's our refund policy for users in Vietnam?
  A: CinemaStream generally follows a 14-day refund policy across all regions....
  Issue: CinemaStream has NO published refund policy document in the warehouse or knowledge base -- the model is generating a plausible-sounding generic SaaS answer, not retrieving a real CinemaStream policy.

Beta failure 2:
  Q: Which country had the highest churn rate last month, and how does that compare to Premium users specifically?
  A: Vietnam had the highest churn rate....
  Issue: The retrieved context DID include a per-plan churn breakdown table, but the model's answer only addressed the first half of the question (highest churn country) and ignored the Premium-specific comparison that was sitting right there in the retrieved rows.

Beta failure 3:
  Q: Give me this week's top 3 movies by completion rate as a JSON list.
  A: Sure! The top 3 movies by completion rate this week are:
1. Monsoon Heart (89%)
2. Hujan di Singapura (84%)
3. Office Hari Ini (78%)...
  Issue: Rohan's downstream Slack-bot integration expects valid JSON to parse and post as a formatted message. This response is human-readable but not valid JSON -- the integration throws a parsing error.
```

In [ ]:
diagnoses = [
    diagnose_gap({"model_lacks_domain_fact": True}),                      # Failure 1
    diagnose_gap({"relevant_info_was_in_context_but_ignored": True}),     # Failure 2
    diagnose_gap({"output_format_is_unparseable": True}),                 # Failure 3
]

for i, (gap_num, gap_name, fix) in enumerate(diagnoses, start=1):
    print(f"Beta failure {i}: Gap {gap_num} ({gap_name})")
    print(f"  Fix: {fix}")

```
Beta failure 1: Gap 1 (Knowledge)
  Fix: RAG -- retrieve the missing fact and add it to context
Beta failure 2: Gap 2 (Context)
  Fix: Context engineering -- restructure what's in the prompt and where
Beta failure 3: Gap 6 (Structure)
  Fix: Output parsing / structured output constraints (e.g., JSON schema, regex validation)
```

In [ ]:
# Before: context buried, question doesn't flag its own multi-part nature
before_prompt = (
    "Here is some general CinemaStream context: [... long block of mixed retrieved data ...]\n"
    "Question: Which country had the highest churn rate last month, and how does that "
    "compare to Premium users specifically?"
)

# After: relevant table placed immediately before the question,
# AND the multi-part nature of the question is made explicit
after_prompt = (
    "Churn rate by country and plan, last month:\n"
    "VN: overall=8.2%, Premium=3.1%\n"
    "PH: overall=6.5%, Premium=5.9%\n"
    "ID: overall=5.0%, Premium=2.0%\n\n"
    "Question has two parts -- answer BOTH explicitly:\n"
    "1. Which country had the highest OVERALL churn rate?\n"
    "2. For that same country, how does its Premium churn rate compare to its overall rate?"
)

print("BEFORE (Gap 2 present):")
print(before_prompt)
print("\nAFTER (context restructured):")
print(after_prompt)

```
BEFORE (Gap 2 present):
Here is some general CinemaStream context: [... long block of mixed retrieved data ...]
Question: Which country had the highest churn rate last month, and how does that compare to Premium users specifically?

AFTER (context restructured):
Churn rate by country and plan, last month:
VN: overall=8.2%, Premium=3.1%
VN: overall=8.2%, Premium=3.1%
ID: overall=5.0%, Premium=2.0%

Question has two parts -- answer BOTH explicitly:
1. Which country had the highest OVERALL churn rate?
2. For that same country, how does its Premium churn rate compare to its overall rate?
```

```
AFTER (context restructured):
Churn rate by country and plan, last month:
VN: overall=8.2%, Premium=3.1%
PH: overall=6.5%, Premium=5.9%
ID: overall=5.0%, Premium=2.0%

Question has two parts -- answer BOTH explicitly:
1. Which country had the highest OVERALL churn rate?
2. For that same country, how does its Premium churn rate compare to its overall rate?
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
gap_num, gap_name, fix = diagnose_gap({"relevant_info_was_in_context_but_ignored": True})
print(f"Gap {gap_num} ({gap_name}): {fix}")

```
Gap 2 (Context): Context engineering -- restructure what's in the prompt and where
```

---

# Chapter 65: Advanced Prompting — KV Cache, Prompt Caching, and the Cost/Latency Math

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install anthropic

### 2.1 The quadratic cost of "no cache" — a toy model

In [ ]:
def cost_without_cache(n_tokens):
    """Total K/V computation work if nothing is cached.
    At step i, recompute K/V for all i tokens seen so far."""
    return sum(range(1, n_tokens + 1))

def cost_with_cache(n_tokens):
    """Total K/V computation work with caching.
    At each step, only the new token's K/V is computed."""
    return n_tokens

for n in [5, 10, 50, 100]:
    wc = cost_without_cache(n)
    c = cost_with_cache(n)
    print(f"n={n:4d} | without cache: {wc:6d} units | with cache: {c:4d} units | ratio: {wc/c:.1f}x")

```
n=   5 | without cache:     15 units | with cache:    5 units | ratio: 3.0x
n=  10 | without cache:     55 units | with cache:   10 units | ratio: 5.5x
n=  50 | without cache:   1275 units | with cache:   50 units | ratio: 25.5x
n= 100 | without cache:   5050 units | with cache:  100 units | ratio: 50.5x
```

### 2.2 Cross-call caching: shared prefixes

In [ ]:
def shared_prefix_len(seq_a, seq_b):
    """How many tokens (represented here as items in a list) match,
    starting from the beginning, before the first mismatch."""
    n = 0
    for x, y in zip(seq_a, seq_b):
        if x == y:
            n += 1
        else:
            break
    return n

# Good ordering: stable content first, variable content last
call1_good = ["SYSTEM_PROMPT", "RAG_CONTEXT_CHURN_TABLE", "USER_QUERY_1"]
call2_good = ["SYSTEM_PROMPT", "RAG_CONTEXT_CHURN_TABLE", "USER_QUERY_2"]

# Bad ordering: variable content first — breaks the prefix match for everything after it
call1_bad = ["USER_QUERY_1", "SYSTEM_PROMPT", "RAG_CONTEXT_CHURN_TABLE"]
call2_bad = ["USER_QUERY_2", "SYSTEM_PROMPT", "RAG_CONTEXT_CHURN_TABLE"]

print("Good ordering shared prefix:", shared_prefix_len(call1_good, call2_good), "of", len(call1_good), "blocks")
print("Bad ordering shared prefix:", shared_prefix_len(call1_bad, call2_bad), "of", len(call1_bad), "blocks")

```
Good ordering shared prefix: 2 of 3 blocks
Bad ordering shared prefix: 0 of 3 blocks
```

### 2.3 Anthropic's prompt caching API

In [ ]:
import os
import anthropic

# Stable system prompt — goes first, gets cached after the first call.
SYSTEM = "You are CinemaStream's data analyst. Answer questions about churn, " \
         "revenue, and engagement using only the data provided in context. " \
         "Always cite the specific numbers you used."  # imagine this is ~2,000 tokens in production

# Variable user query — goes last, never cached.
user_query = "Which country had the highest churn last week?"

# This is a LIVE API call, so it needs ANTHROPIC_API_KEY. Without a key (e.g., CI),
# skip the call and show the illustrative first-call usage instead.
if os.environ.get("ANTHROPIC_API_KEY"):
    client = anthropic.Anthropic()
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system=[{
            "type": "text",
            "text": SYSTEM,
            "cache_control": {"type": "ephemeral"}  # tell Claude to cache this block
        }],
        messages=[{"role": "user", "content": user_query}]
    )
    print(response.usage)
else:
    print("Usage(input_tokens=12, cache_creation_input_tokens=187, cache_read_input_tokens=0, output_tokens=84)")

```
Usage(input_tokens=12, cache_creation_input_tokens=187, cache_read_input_tokens=0, output_tokens=84)
```

```
Usage(input_tokens=12, cache_creation_input_tokens=0, cache_read_input_tokens=187, output_tokens=91)
```

### 2.4 The cost math

In [ ]:
# Illustrative per-million-token pricing (order of magnitude, not exact current rates)
base_input_price = 3.00     # USD per million input tokens
cache_write_price = 3.75    # USD per million tokens, first time written to cache (+25%)
cache_read_price = 0.30     # USD per million tokens, served from cache (-90%)

system_tokens = 2000
calls_per_day = 500

# Without caching: every call pays full price for the system prompt tokens
cost_no_cache = (system_tokens * calls_per_day / 1_000_000) * base_input_price

# With caching: first call pays cache-write price, remaining calls pay cache-read price
cost_with_cache = (
    (system_tokens / 1_000_000) * cache_write_price
    + (system_tokens * (calls_per_day - 1) / 1_000_000) * cache_read_price
)

savings_pct = (1 - cost_with_cache / cost_no_cache) * 100

print(f"No cache:   ${cost_no_cache:.4f}/day")
print(f"With cache: ${cost_with_cache:.4f}/day")
print(f"Savings:    {savings_pct:.1f}%")
print(f"Monthly savings (30 days): ${(cost_no_cache - cost_with_cache) * 30:.2f}")

```
No cache:   $3.0000/day
With cache: $0.3069/day
Savings:    89.8%
Monthly savings (30 days): $80.79
```

### 2.5 The latency side: TTFT, not just cost

In [ ]:
# Illustrative: prefill cost per 1,000 tokens, and per-token output generation cost
prefill_ms_per_1k_tokens = 100
output_ms_per_token = 30

system_tokens = 2000
output_tokens = 200

# Uncached: full prefill cost for the system prompt
prefill_uncached = (system_tokens / 1000) * prefill_ms_per_1k_tokens
gen_time = output_tokens * output_ms_per_token
total_uncached = prefill_uncached + gen_time

# Cached: assume cache-read prefill is ~10% of normal prefill time
prefill_cached = prefill_uncached * 0.1
total_cached = prefill_cached + gen_time

print(f"Uncached: prefill={prefill_uncached:.0f}ms, total={total_uncached:.0f}ms")
print(f"Cached:   prefill={prefill_cached:.0f}ms, total={total_cached:.0f}ms")
print(f"Latency reduction: {(1 - total_cached / total_uncached) * 100:.1f}%")

```
Uncached: prefill=200ms, total=6200ms
Cached:   prefill=20ms, total=6020ms
Latency reduction: 2.9%
```

### 2.6 A new kind of "gap": Gap 0, the efficiency gap

In [ ]:
def diagnose_efficiency(prompt_blocks, calls_per_day):
    """A toy efficiency check: is the cacheable content positioned correctly,
    and is volume high enough that caching matters?"""
    stable_blocks = [b for b in prompt_blocks if b["stable"]]
    variable_blocks = [b for b in prompt_blocks if not b["stable"]]

    # Cache-friendly ordering: ALL stable blocks must come before ANY variable block
    stable_positions = [i for i, b in enumerate(prompt_blocks) if b["stable"]]
    variable_positions = [i for i, b in enumerate(prompt_blocks) if not b["stable"]]
    cache_friendly_order = (
        bool(stable_positions) and bool(variable_positions)
        and max(stable_positions) < min(variable_positions)
    )

    cacheable_tokens = sum(b["tokens"] for b in stable_blocks)
    volume_threshold = 50  # below this, caching overhead may not be worth it

    if not stable_blocks:
        return "No stable content to cache — Gap 0 not applicable."
    if not cache_friendly_order:
        return f"Gap 0 (Efficiency): {cacheable_tokens} cacheable tokens exist, " \
               f"but prompt ordering breaks the prefix match. Fix: move all stable " \
               f"blocks before variable blocks."
    if calls_per_day < volume_threshold:
        return f"Gap 0 (low priority): ordering is cache-friendly, but only " \
               f"{calls_per_day} calls/day — savings may not justify added complexity yet."
    return f"Gap 0: well-positioned. {cacheable_tokens} tokens cacheable across " \
           f"{calls_per_day} calls/day — apply cache_control."

# Example: CinemaStream's "Ask Anything" tool
prompt_blocks = [
    {"name": "system_instructions", "stable": True, "tokens": 600},
    {"name": "rag_context_churn_table", "stable": True, "tokens": 1400},
    {"name": "user_query", "stable": False, "tokens": 25},
]

print(diagnose_efficiency(prompt_blocks, calls_per_day=500))

```
Gap 0: well-positioned. 2000 tokens cacheable across 500 calls/day — apply cache_control.
```

## 3. CinemaStream in Practice

In [ ]:
def build_prompt(system_instructions, retrieved_context, user_question):
    """Current (Chapter 061 / 064) prompt construction for Ask Anything."""
    return (
        f"{system_instructions}\n\n"
        f"Retrieved context:\n{retrieved_context}\n\n"
        f"Question: {user_question}"
    )

# CinemaStream's actual stable content (illustrative token counts)
SYSTEM_INSTRUCTIONS = (
    "You are CinemaStream's internal analytics assistant. CinemaStream is a "
    "video streaming startup serving SG, MY, ID, PH, TH, VN, IN. Plans are "
    "Free (S$0), Basic (S$8.90), Premium (S$12.90). Answer using only the "
    "context provided. If the answer requires data not present in the "
    "context, say so explicitly rather than guessing. Cite specific numbers."
)  # ~600 tokens in production

RAG_CONTEXT = (
    "Churn rate by country and plan, last month:\n"
    "VN: overall=8.2%, Premium=3.1%\n"
    "PH: overall=6.5%, Premium=5.9%\n"
    "ID: overall=5.0%, Premium=2.0%\n"
    "Active users: 100 (56 Free / 28 Basic / 16 Premium)\n"
    "MRR: S$363.50"
)  # ~1400 tokens in production with the full table

example_prompt = build_prompt(SYSTEM_INSTRUCTIONS, RAG_CONTEXT, "What's our MRR?")
print(example_prompt)

```
You are CinemaStream's internal analytics assistant. CinemaStream is a video streaming startup serving SG, MY, ID, PH, TH, VN, IN. Plans are Free (S$0), Basic (S$8.90), Premium (S$12.90). Answer using only the context provided. If the answer requires data not present in the context, say so explicitly rather than guessing. Cite specific numbers.

Retrieved context:
Churn rate by country and plan, last month:
VN: overall=8.2%, Premium=3.1%
PH: overall=6.5%, Premium=5.9%
ID: overall=5.0%, Premium=2.0%
Active users: 100 (56 Free / 28 Basic / 16 Premium)
MRR: S$363.50

Question: What's our MRR?
```

In [ ]:
prompt_blocks = [
    {"name": "system_instructions", "stable": True, "tokens": 600},
    {"name": "rag_context", "stable": True, "tokens": 1400},
    {"name": "user_question", "stable": False, "tokens": 15},
]
print(diagnose_efficiency(prompt_blocks, calls_per_day=480))

```
Gap 0: well-positioned. 2000 tokens cacheable across 480 calls/day — apply cache_control.
```

In [ ]:
base_input_price = 3.00
cache_write_price = 3.75
cache_read_price = 0.30

cacheable_tokens = 2000
calls_per_day = 480

cost_no_cache = (cacheable_tokens * calls_per_day / 1_000_000) * base_input_price
cost_with_cache = (
    (cacheable_tokens / 1_000_000) * cache_write_price
    + (cacheable_tokens * (calls_per_day - 1) / 1_000_000) * cache_read_price
)

print(f"No cache:   ${cost_no_cache:.2f}/day -> ${cost_no_cache * 30:.2f}/month")
print(f"With cache: ${cost_with_cache:.2f}/day -> ${cost_with_cache * 30:.2f}/month")
print(f"Monthly savings: ${(cost_no_cache - cost_with_cache) * 30:.2f}")

```
No cache:   $2.88/day -> $86.40/month
With cache: $0.29/day -> $8.85/month
Monthly savings: $77.55
```

In [ ]:
import anthropic

def ask_anything_cached(system_instructions, retrieved_context, user_question, client=None):
    """Chapter 065 update to ask_anything(): adds cache_control to the
    stable system+context block. The function signature and return shape
    are UNCHANGED from Chapter 061/064 -- only the API call construction
    changes internally."""
    client = client or anthropic.Anthropic()

    cacheable_block = (
        f"{system_instructions}\n\n"
        f"Retrieved context:\n{retrieved_context}"
    )

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system=[{
            "type": "text",
            "text": cacheable_block,
            "cache_control": {"type": "ephemeral"}
        }],
        messages=[{"role": "user", "content": user_question}]
    )
    return response

# Demonstrating the prefix-ordering check still holds for two different questions
call_a = ["SYSTEM_INSTRUCTIONS+RAG_CONTEXT", "What's our MRR?"]
call_b = ["SYSTEM_INSTRUCTIONS+RAG_CONTEXT", "Which country has the highest churn?"]
print("Shared prefix blocks:", shared_prefix_len(call_a, call_b), "of", len(call_a))

```
Shared prefix blocks: 1 of 2
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
def build_prompt(user_id, system_instructions, context):
    return f"User {user_id} asks at request time.\n\n{system_instructions}\n\n{context}\n\nAnswer the question above."

In [ ]:
def build_prompt_fixed(user_id, system_instructions, context):
    # Stable content first (this is what gets cache_control)
    stable = f"{system_instructions}\n\n{context}"
    # Variable content last
    variable = f"\n\nUser {user_id} asks at request time. Answer the question above."
    return stable, variable

stable, variable = build_prompt_fixed(42, "SYSTEM_INSTRUCTIONS", "CONTEXT_BLOCK")
print("Stable (cacheable):", stable)
print("Variable (never cached):", variable)

```
Stable (cacheable): SYSTEM_INSTRUCTIONS

CONTEXT_BLOCK
Variable (never cached): 

User 42 asks at request time. Answer the question above.
```

In [ ]:
prompt_blocks = [
    {"name": "personalized_greeting", "stable": False, "tokens": 30},
    {"name": "system_instructions", "stable": True, "tokens": 600},
    {"name": "rag_context", "stable": True, "tokens": 1400},
    {"name": "user_question", "stable": False, "tokens": 15},
]
print(diagnose_efficiency(prompt_blocks, calls_per_day=480))

```
Gap 0 (Efficiency): 2000 cacheable tokens exist, but prompt ordering breaks the prefix match. Fix: move all stable blocks before variable blocks.
```

---

# Chapter 66: LLM Workflow Automation — LangChain, Hybrid RAG, and MCP Servers

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy

### 2.1 LangChain core concepts: chains and prompt templates

In [ ]:
class PromptTemplate:
    """Minimal reimplementation of LangChain's PromptTemplate concept --
    a parameterized prompt string with named placeholders."""
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self, **kwargs):
        missing = set(self.input_variables) - set(kwargs)
        if missing:
            raise ValueError(f"Missing required variables: {missing}")
        return self.template.format(**kwargs)

summary_prompt = PromptTemplate(
    template="Summarize the following {data_type} data in 2 sentences:\n\n{data}",
    input_variables=["data_type", "data"],
)

filled = summary_prompt.format(
    data_type="watch_events",
    data="381 events, 100 users, avg watch time 87 minutes"
)
print(filled)

```
Summarize the following watch_events data in 2 sentences:

381 events, 100 users, avg watch time 87 minutes
```

In [ ]:
def simulated_llm(prompt):
    """Stand-in for an actual model call -- returns a deterministic
    'response' so this example runs without an API key."""
    if "Summarize" in prompt:
        return "Watch activity is healthy: 381 events across 100 users, averaging 87 minutes per session."
    if "translate" in prompt.lower():
        return "[ID translation] Aktivitas menonton sehat: 381 acara di 100 pengguna."
    return "[unhandled prompt]"

class SequentialChain:
    """Minimal reimplementation: runs a list of (template, llm) steps,
    feeding each step's output into the next step's input variable."""
    def __init__(self, steps, output_variable="text"):
        self.steps = steps  # list of (PromptTemplate, output_key)
        self.output_variable = output_variable

    def run(self, **initial_inputs):
        context = dict(initial_inputs)
        for template, output_key in self.steps:
            prompt = template.format(**context)
            context[output_key] = simulated_llm(prompt)
        return context

translate_prompt = PromptTemplate(
    template="translate to Bahasa Indonesia: {summary}",
    input_variables=["summary"],
)

chain = SequentialChain(steps=[
    (summary_prompt, "summary"),
    (translate_prompt, "translation"),
])

result = chain.run(data_type="watch_events", data="381 events, 100 users, avg watch time 87 minutes")
print("Step 1 output (summary):", result["summary"])
print("Step 2 output (translation):", result["translation"])

```
Step 1 output (summary): Watch activity is healthy: 381 events across 100 users, averaging 87 minutes per session.
Step 2 output (translation): [ID translation] Aktivitas menonton sehat: 381 acara di 100 pengguna.
```

### 2.2 Memory: carrying context across turns

In [ ]:
class ConversationBufferMemory:
    """Minimal reimplementation: stores (role, content) turns and renders
    them as a transcript to prepend to the next prompt."""
    def __init__(self):
        self.turns = []

    def add_turn(self, role, content):
        self.turns.append((role, content))

    def render(self):
        return "\n".join(f"{role}: {content}" for role, content in self.turns)

memory = ConversationBufferMemory()
memory.add_turn("user", "What's our churn rate?")
memory.add_turn("assistant", "Overall churn rate is 24.0%.")
memory.add_turn("user", "And just for Vietnam?")

prompt_with_history = f"{memory.render()}\nassistant:"
print(prompt_with_history)

```
user: What's our churn rate?
assistant: Overall churn rate is 24.0%.
user: And just for Vietnam?
assistant:
```

### 2.3 Tools and the agent loop (ReAct pattern)

In [ ]:
def get_active_users_count(country=None):
    """A 'tool' -- a real function the agent can call."""
    data = {"SG": 12, "MY": 18, "ID": 22, "PH": 15, "TH": 11, "VN": 14, "IN": 8}
    if country:
        return {country: data.get(country, 0)}
    return {"total": sum(data.values()), "by_country": data}

TOOLS = {
    "get_active_users_count": get_active_users_count,
}

def agent_loop(question, max_steps=3):
    """Toy ReAct loop: a 'reasoning' step decides whether a tool is needed,
    based on simple keyword matching (a real agent uses the LLM for this)."""
    steps = []
    if "active users" in question.lower():
        country = None
        for code in ["SG", "MY", "ID", "PH", "TH", "VN", "IN"]:
            if code.lower() in question.lower():
                country = code
        steps.append(("Reason", f"Question asks about active users (country={country}). Need to call get_active_users_count."))
        result = TOOLS["get_active_users_count"](country=country)
        steps.append(("Act", f"get_active_users_count(country={country!r}) -> {result}"))
        steps.append(("Observe", f"Tool returned: {result}"))
        if country:
            answer = f"{country} has {result[country]} active users."
        else:
            answer = f"Total active users: {result['total']} ({result['by_country']})"
        steps.append(("Answer", answer))
    else:
        steps.append(("Answer", "I don't have a tool for that question."))
    return steps

for label, content in agent_loop("How many active users does VN have?"):
    print(f"{label}: {content}")

```
Reason: Question asks about active users (country=VN). Need to call get_active_users_count.
Act: get_active_users_count(country='VN') -> {'VN': 14}
Observe: Tool returned: {'VN': 14}
Answer: VN has 14 active users.
```

### 2.4 Hybrid RAG: the four-level maturity ladder

In [ ]:
import math
from collections import Counter

documents = [
    "Monsoon Heart is a 2024 Hindi drama set in coastal Kerala about a monsoon love story.",
    "Hujan di Singapura is a 2023 Malay thriller about a Singapore cybercrime unit racing a deadline.",
    "Office Hari Ini is a 2024 Indonesian comedy about a Jakarta ad agency adjusting to remote work.",
    "Kayal Veedu is a 2019 Malayalam family drama about three generations in a fishing village in Kerala.",
    "Singapore Nights is a 2022 English drama about expatriate life and family secrets in Singapore.",
]

def tokenize(text):
    return text.lower().replace(",", "").replace(".", "").split()

tokenized_docs = [tokenize(d) for d in documents]
N = len(tokenized_docs)
avgdl = sum(len(d) for d in tokenized_docs) / N
df = Counter()
for d in tokenized_docs:
    for term in set(d):
        df[term] += 1

k1, b = 1.5, 0.75  # standard BM25 tuning constants

def bm25_score(query_tokens, doc_tokens):
    score = 0.0
    tf = Counter(doc_tokens)
    dl = len(doc_tokens)
    for term in query_tokens:
        if term not in tf:
            continue
        idf = math.log((N - df[term] + 0.5) / (df[term] + 0.5) + 1)
        freq = tf[term]
        score += idf * (freq * (k1 + 1)) / (freq + k1 * (1 - b + b * dl / avgdl))
    return score

def bm25_search(query, top_k=3):
    q_tokens = tokenize(query)
    scores = [(i, bm25_score(q_tokens, tokenized_docs[i])) for i in range(N)]
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

query = "Singapore drama Malayalam"
for i, score in bm25_search(query):
    print(f"  [{i}] score={score:.3f} -- {documents[i][:55]}...")

```
  [3] score=1.883 -- Kayal Veedu is a 2019 Malayalam family drama about thr...
  [4] score=1.839 -- Singapore Nights is a 2022 English drama about expatri...
  [1] score=0.880 -- Hujan di Singapura is a 2023 Malay thriller about a Sin...
```

In [ ]:
import numpy as np

vocab = sorted(set(t for d in tokenized_docs for t in d))
vocab_index = {t: i for i, t in enumerate(vocab)}

def vectorize(tokens):
    v = np.zeros(len(vocab))
    for t in tokens:
        if t in vocab_index:
            v[vocab_index[t]] += 1
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

doc_vectors = np.array([vectorize(d) for d in tokenized_docs])

def vector_search(query, top_k=3):
    q_vec = vectorize(tokenize(query))
    sims = doc_vectors @ q_vec
    ranked = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i])) for i in ranked]

for i, score in vector_search(query):
    print(f"  [{i}] score={score:.3f} -- {documents[i][:55]}...")

```
  [4] score=0.420 -- Singapore Nights is a 2022 English drama about expatri...
  [3] score=0.252 -- Kayal Veedu is a 2019 Malayalam family drama about thr...
  [0] score=0.129 -- Monsoon Heart is a 2024 Hindi drama set in coastal Ker...
```

In [ ]:
def rrf(rankings, k=60):
    """Reciprocal Rank Fusion: each document gets 1/(k + rank + 1) from
    each ranking it appears in; scores are summed across rankings."""
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

bm25_ids = [i for i, _ in bm25_search(query)]
vector_ids = [i for i, _ in vector_search(query)]
fused = rrf([bm25_ids, vector_ids])

print("BM25 ranking:  ", bm25_ids)
print("Vector ranking:", vector_ids)
print("RRF fused:     ", fused)
print("\nTop result after fusion:", documents[fused[0]])

```
BM25 ranking:   [3, 4, 1]
Vector ranking: [4, 3, 0]
RRF fused:      [3, 4, 1, 0]
```

```
Top result after fusion: Kayal Veedu is a 2019 Malayalam family drama about three generations in a fishing village in Kerala.
```

### 2.5 MCP: building a tool server

In [ ]:
import json

USERS = [
    {"user_id": 1, "name": "Ravi Kumar", "country": "IN", "plan": "Premium"},
    {"user_id": 2, "name": "Siti Rahman", "country": "MY", "plan": "Basic"},
    {"user_id": 3, "name": "Nguyen Van Minh", "country": "VN", "plan": "Free"},
]
WATCH_EVENTS = [
    {"user_id": 1, "movie_id": 101, "watch_minutes": 132, "country": "IN"},
    {"user_id": 1, "movie_id": 102, "watch_minutes": 45, "country": "IN"},
    {"user_id": 2, "movie_id": 103, "watch_minutes": 96, "country": "MY"},
]

TOOLS = {
    "query_watch_events": {
        "description": "Query CinemaStream watch_events table. Returns viewing stats by country.",
        "input_schema": {"country": "string (optional)"},
    },
    "get_churn_risk_users": {
        "description": "Returns users with no watch events at all (proxy for high churn risk).",
        "input_schema": {},
    },
}

def list_tools():
    return TOOLS

def call_tool(name, arguments=None):
    arguments = arguments or {}
    if name == "query_watch_events":
        country = arguments.get("country")
        events = [e for e in WATCH_EVENTS if country is None or e["country"] == country]
        total_minutes = sum(e["watch_minutes"] for e in events)
        return json.dumps({"country": country, "event_count": len(events), "total_watch_minutes": total_minutes})
    elif name == "get_churn_risk_users":
        active_user_ids = {e["user_id"] for e in WATCH_EVENTS}
        at_risk = [u for u in USERS if u["user_id"] not in active_user_ids]
        return json.dumps(at_risk)
    raise ValueError(f"Unknown tool: {name}")

print("Available tools:")
for name, spec in list_tools().items():
    print(f"  {name}: {spec['description']}")

print("\nQuestion: 'Which users haven't watched anything at all?'")
print("(Claude maps this to get_churn_risk_users() with no arguments)")
print("Tool result:", call_tool("get_churn_risk_users"))

print("\nQuestion: 'How much watch time came from India?'")
print("(Claude maps this to query_watch_events(country='IN'))")
print("Tool result:", call_tool("query_watch_events", {"country": "IN"}))

```
Available tools:
  query_watch_events: Query CinemaStream watch_events table. Returns viewing stats by country.
  get_churn_risk_users: Returns users with no watch events at all (proxy for high churn risk).

Question: 'Which users haven't watched anything at all?'
(Claude maps this to get_churn_risk_users() with no arguments)
Tool result: [{"user_id": 3, "name": "Nguyen Van Minh", "country": "VN", "plan": "Free"}]

Question: 'How much watch time came from India?'
(Claude maps this to query_watch_events(country='IN'))
Tool result: {"country": "IN", "event_count": 2, "total_watch_minutes": 177}
```

### 2.6 LLMs inside Airflow DAGs: error handling and fallbacks

In [ ]:
class LLMUnavailableError(Exception):
    pass

def call_llm(prompt, _force_fail_first_n=0, _attempt=[0]):
    """Simulated LLM call -- fails the first _force_fail_first_n times,
    then succeeds. Mimics a transient outage."""
    _attempt[0] += 1
    if _attempt[0] <= _force_fail_first_n:
        raise LLMUnavailableError(f"503 Service Unavailable (attempt {_attempt[0]})")
    return f"Summary: {prompt[:30]}... [generated]"

def summarize_with_fallback(prompt, max_retries=3):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return call_llm(prompt, _force_fail_first_n=2)
        except LLMUnavailableError as e:
            last_error = e
            print(f"  attempt {attempt} failed: {e}")
    print(f"  all {max_retries} attempts failed ({last_error}), using fallback")
    return f"Summary unavailable (LLM error). Raw excerpt: {prompt[:50]}..."

prompt = "Daily watch_events summary for 2026-06-09: 381 events across 100 users..."
result = summarize_with_fallback(prompt)
print("\nFinal result:", result)

```
  attempt 1 failed: 503 Service Unavailable (attempt 1)
  attempt 2 failed: 503 Service Unavailable (attempt 2)

Final result: Summary: Daily watch_events summary for... [generated]
```

## 3. CinemaStream in Practice

In [ ]:
kb_documents = [
    "Plan Pricing: Free is S$0/month, Basic is S$8.90/month, Premium is S$12.90/month.",
    "Refund Policy: CinemaStream does not currently offer refunds for partial billing periods.",
    "Account Deletion: Users can delete their account from Settings > Account > Delete Account.",
    "Premium Plan VN Churn: Vietnam Premium-tier churn rate is 3.1%, notably lower than VN's overall 8.2% churn rate.",
    "Data Retention: Raw event data is retained for 90 days in the warehouse's raw layer.",
]

query = "What's the Premium churn situation in Vietnam specifically?"

# naive RAG: bag-of-words vector search only (Ch061's approach)
def tokenize(text):
    return text.lower().replace(",", "").replace(".", "").replace(":", "").split()

tokenized_kb = [tokenize(d) for d in kb_documents]
vocab = sorted(set(t for d in tokenized_kb for t in d))
vocab_index = {t: i for i, t in enumerate(vocab)}

def vectorize(tokens):
    v = np.zeros(len(vocab))
    for t in tokens:
        if t in vocab_index:
            v[vocab_index[t]] += 1
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

kb_vectors = np.array([vectorize(d) for d in tokenized_kb])

def vector_search_kb(q, top_k=2):
    qv = vectorize(tokenize(q))
    sims = kb_vectors @ qv
    ranked = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i])) for i in ranked]

print("Naive RAG (vector-only) results:")
for i, score in vector_search_kb(query):
    print(f"  score={score:.3f} -- {kb_documents[i]}")

```
Naive RAG (vector-only) results:
  score=0.439 -- Premium Plan VN Churn: Vietnam Premium-tier churn rate is 3.1%, notably lower than VN's overall 8.2% churn rate.
  score=0.205 -- Data Retention: Raw event data is retained for 90 days in the warehouse's raw layer.
```

In [ ]:
results = vector_search_kb(query)
print("Actual order returned:", [(i, round(s, 3)) for i, s in results])
for i, score in results:
    print(f"  score={score:.3f} -- {kb_documents[i][:60]}...")

```
Actual order returned: [(3, 0.439), (4, 0.205)]
  score=0.439 -- Premium Plan VN Churn: Vietnam Premium-tier churn rate is 3....
  score=0.205 -- Data Retention: Raw event data is retained for 90 days in th...
```

In [ ]:
rohan_query = "that figure Mei quoted, ours was better right?"
results = vector_search_kb(rohan_query)
print("Naive RAG results for Rohan's actual phrasing:")
for i, score in results:
    print(f"  score={score:.3f} -- {kb_documents[i][:60]}...")

```
Naive RAG results for Rohan's actual phrasing:
  score=0.000 -- Data Retention: Raw event data is retained for 90 days in th...
  score=0.000 -- Premium Plan VN Churn: Vietnam Premium-tier churn rate is 3....
```

In [ ]:
def rewrite_query(raw_query, conversation_context):
    """L3: rewrite an ambiguous query using prior conversation context.
    In production, this is itself an LLM call with the conversation
    history (Section 2.2's ConversationBufferMemory) as context."""
    if "figure" in raw_query.lower() and "better" in raw_query.lower():
        return "Vietnam Premium churn rate lower than overall"
    return raw_query

conversation = ConversationBufferMemory()
conversation.add_turn("user", "What's our churn breakdown by country and plan?")
conversation.add_turn("assistant", "VN: overall=8.2%, Premium=3.1%. PH: overall=6.5%, Premium=5.9%. ID: overall=5.0%, Premium=2.0%.")

rewritten = rewrite_query(rohan_query, conversation.render())
print("Original: ", rohan_query)
print("Rewritten:", rewritten)

results = vector_search_kb(rewritten)
print("\nResults after rewriting:")
for i, score in results:
    print(f"  score={score:.3f} -- {kb_documents[i][:60]}...")

```
Original:  that figure Mei quoted, ours was better right?
Rewritten: Vietnam Premium churn rate lower than overall

Results after rewriting:
  score=0.741 -- Premium Plan VN Churn: Vietnam Premium-tier churn rate is 3....
  score=0.092 -- Plan Pricing: Free is S$0/month, Basic is S$8.90/month, Prem...
```

In [ ]:
def get_churn_by_country_and_plan(country=None, plan=None):
    """MCP-style tool: queries LIVE data instead of a static KB document.
    Mirrors the canonical Ch064 churn-by-country-and-plan figures."""
    data = {
        ("VN", "overall"): 8.2, ("VN", "Premium"): 3.1,
        ("PH", "overall"): 6.5, ("PH", "Premium"): 5.9,
        ("ID", "overall"): 5.0, ("ID", "Premium"): 2.0,
    }
    if country and plan:
        return {f"{country}_{plan}_churn_pct": data.get((country, plan), "no data")}
    return {f"{c}_{p}": v for (c, p), v in data.items() if c == country} if country else data

print("MCP tool call: get_churn_by_country_and_plan(country='VN', plan='Premium')")
print(get_churn_by_country_and_plan(country="VN", plan="Premium"))

```
MCP tool call: get_churn_by_country_and_plan(country='VN', plan='Premium')
{'VN_Premium_churn_pct': 3.1}
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
ranking_a = ["doc5", "doc2", "doc9"]
ranking_b = ["doc2", "doc9", "doc5"]

In [ ]:
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

ranking_a = ["doc5", "doc2", "doc9"]
ranking_b = ["doc2", "doc9", "doc5"]

fused = rrf([ranking_a, ranking_b])
for doc in fused:
    print(doc)

```
doc2
doc5
doc9
```

---

# Chapter 66a: Statistics & Probability Foundations

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Descriptive Statistics

In [ ]:
import statistics
import math


data = [12, 45, 23, 67, 34, 89, 11, 56, 78, 34, 23, 45, 12, 90, 56]

mean   = statistics.mean(data)
median = statistics.median(data)
mode   = statistics.mode(data)
stdev  = statistics.stdev(data)       # sample std dev (divides by n-1)
var    = statistics.variance(data)    # sample variance

print(f"Mean:   {mean:.2f}")
print(f"Median: {median:.2f}")
print(f"Mode:   {mode}")
print(f"Std:    {stdev:.2f}")
print(f"Var:    {var:.2f}")

# Range and IQR
data_sorted = sorted(data)
q1 = statistics.quantiles(data, n=4)[0]   # 25th percentile
q3 = statistics.quantiles(data, n=4)[2]   # 75th percentile
iqr = q3 - q1
print(f"\nMin: {min(data)}, Max: {max(data)}, Range: {max(data)-min(data)}")
print(f"Q1: {q1:.1f}, Q3: {q3:.1f}, IQR: {iqr:.1f}")

In [ ]:
import statistics


# Skewness intuition — compare mean vs median
right_skewed        = [1, 2, 2, 3, 3, 3, 4, 5, 100]   # one big outlier
slightly_right_skewed = [3, 4, 5, 5, 5, 6, 7, 8, 9]      # mild rightward pull

for name, d in [("right_skewed", right_skewed), ("slightly_right_skewed", slightly_right_skewed)]:
    m  = statistics.mean(d)
    md = statistics.median(d)
    print(f"{name}: mean={m:.1f}, median={md:.1f}, "
          f"direction={'right-skewed (mean>median)' if m > md else 'symmetric/left'}")

### 2.2 Probability Basics

In [ ]:
# CinemaStream: 100 subscribers, plan breakdown
total  = 100
free   = 56
basic  = 28
prem   = 16
churned_free  = 17    # 30.4% of 56
churned_basic = 5     # 17.9% of 28
churned_prem  = 2     # 12.5% of 16

# P(churned) = overall churn rate
p_churned = (churned_free + churned_basic + churned_prem) / total
print(f"P(churned)           = {p_churned:.3f}")

# P(Free) = probability a random subscriber is on Free plan
p_free = free / total
print(f"P(Free plan)         = {p_free:.3f}")

# P(churned AND Free) = joint probability
p_churned_and_free = churned_free / total
print(f"P(churned ∩ Free)    = {p_churned_and_free:.3f}")

# P(churned | Free) = conditional probability
# "Given a subscriber is on Free plan, what is P(they churn)?"
p_churned_given_free = churned_free / free
print(f"P(churned | Free)    = {p_churned_given_free:.3f}")

# P(churned | Premium)
p_churned_given_prem = churned_prem / prem
print(f"P(churned | Premium) = {p_churned_given_prem:.3f}")

# Bayes check: P(Free | churned) using Bayes' theorem
# P(A|B) = P(B|A) * P(A) / P(B)
p_free_given_churned_bayes = (p_churned_given_free * p_free) / p_churned
p_free_given_churned_direct = churned_free / (churned_free + churned_basic + churned_prem)
print(f"\nP(Free | churned) via Bayes:  {p_free_given_churned_bayes:.3f}")
print(f"P(Free | churned) direct:     {p_free_given_churned_direct:.3f}")
print("(They match — Bayes' theorem is just algebra.)")

### 2.3 Common Distributions

In [ ]:
import math


def normal_pdf(x: float, mu: float, sigma: float) -> float:
    """Probability density at x for N(mu, sigma^2)."""
    return (1 / (sigma * math.sqrt(2 * math.pi))) * math.exp(-0.5 * ((x - mu) / sigma) ** 2)


# Model probability scores from Ch073 (tuned RF) approximate a normal
# distribution in the non-churned population around ~0.15
mu, sigma = 0.15, 0.07
for x in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    density = normal_pdf(x, mu, sigma)
    print(f"  p(score={x:.2f}) density = {density:.4f}")

# 68-95-99.7 rule
print(f"\n68% interval: [{mu - sigma:.2f}, {mu + sigma:.2f}]")
print(f"95% interval: [{mu - 2*sigma:.2f}, {mu + 2*sigma:.2f}]")

In [ ]:
import math


def binomial_prob(n: int, k: int, p: float) -> float:
    """P(exactly k successes in n trials with success prob p)."""
    comb = math.comb(n, k)
    return comb * (p ** k) * ((1 - p) ** (n - k))


# Out of 28 Basic-plan subscribers, how many will churn?
# Each has P(churn) = 0.179 (17.9% from Section 2.2)
n, p = 28, 0.179
print(f"Binomial(n={n}, p={p:.3f})")
print(f"Expected churned: {n * p:.1f}")
print()
for k in range(0, 12):
    prob = binomial_prob(n, k, p)
    bar  = "█" * round(prob * 100)
    print(f"  k={k:2d}: P={prob:.4f}  {bar}")

In [ ]:
import math


def poisson_prob(k: int, lam: float) -> float:
    """P(exactly k events) when the average rate is lam."""
    return (lam ** k * math.exp(-lam)) / math.factorial(k)


# Support tickets: on average 3.2 per day.
# How likely is 0, 1, 2, ... tickets tomorrow?
lam = 3.2
print(f"Poisson(λ={lam})")
for k in range(0, 10):
    prob = poisson_prob(k, lam)
    bar  = "█" * round(prob * 100)
    print(f"  k={k}: P={prob:.4f}  {bar}")

### 2.4 Bayes' Theorem — the update

In [ ]:
# Bayesian churn intuition:
# Prior: a random subscriber has P(churn) = 0.24
# Evidence: subscriber hasn't watched in 30 days
# Likelihood: P(inactive 30d | churned) = 0.82  (from Ch072 eval slice)
# Likelihood: P(inactive 30d | not churned) = 0.05  (most actives watched recently)

p_churn_prior   = 0.24
p_nochurn_prior = 1 - p_churn_prior

p_inactive_given_churn    = 0.82
p_inactive_given_nochurn  = 0.05

# P(inactive) = law of total probability
p_inactive = (p_inactive_given_churn  * p_churn_prior +
              p_inactive_given_nochurn * p_nochurn_prior)

# P(churn | inactive) = Bayes update
p_churn_given_inactive = (p_inactive_given_churn * p_churn_prior) / p_inactive

print(f"Prior P(churn):                     {p_churn_prior:.3f}")
print(f"P(inactive | churn):                {p_inactive_given_churn:.3f}")
print(f"P(inactive | not churn):            {p_inactive_given_nochurn:.3f}")
print(f"P(inactive):                        {p_inactive:.3f}")
print(f"Posterior P(churn | inactive):      {p_churn_given_inactive:.3f}")
print(f"\nSeeing 'inactive 30d' raises churn probability "
      f"from {p_churn_prior:.0%} to {p_churn_given_inactive:.0%}.")

## 3. CinemaStream in Practice

In [ ]:
import statistics
import math


# Canonical CinemaStream dashboard dataset (100 users, seed=42, from Ch060)
# Watch minutes from the synthetic dataset (representative sample)
watch_minutes = [
    0,  0,  12,  23,  45,  67,  89,  34,  56,  78,
    12, 23,  45,   0,  90,  56, 134, 210,  23,  45,
    67, 89,  12,  34,  56,  78,  90,   0,  12,  34,
    45, 23,   0,  67,  89, 134,  56,  23,  45,  78,
    90, 12,  34,  56,   0,  23,  45,  67,  89,  12,
]

# Plan breakdown from Ch060: 56 Free, 28 Basic, 16 Premium
plans    = {"Free": 56, "Basic": 28, "Premium": 16}
churned  = {"Free": 17, "Basic": 5,  "Premium": 2}
total    = 100

print("=== Descriptive Statistics: Watch Minutes ===")
mean_wm = statistics.mean(watch_minutes)
med_wm  = statistics.median(watch_minutes)
std_wm  = statistics.stdev(watch_minutes)
print(f"Mean:   {mean_wm:.1f} min")
print(f"Median: {med_wm:.1f} min")
print(f"Std:    {std_wm:.1f} min")
print(f"(Mean > Median → right-skewed: a few heavy viewers pull the mean up)")

print("\n=== Churn Probability by Plan ===")
for plan in ["Free", "Basic", "Premium"]:
    n = plans[plan]
    c = churned[plan]
    p = c / n
    expected_churned = n * p
    # Std of a Binomial: sqrt(n * p * (1-p))
    binomial_std = math.sqrt(n * p * (1 - p))
    print(f"  {plan:8s}: P(churn)={p:.3f}  "
          f"expected={expected_churned:.1f} ± {binomial_std:.1f} (±1 std)")

print("\n=== Overall Churn Rate ===")
total_churned = sum(churned.values())
p_overall = total_churned / total
print(f"P(churn) = {total_churned}/{total} = {p_overall:.3f}")

print("\n=== Bayes Update: High-Value Premium Subscriber, Inactive 30d ===")
p_prior = churned["Premium"] / plans["Premium"]   # 0.125
p_inactive_given_churn   = 0.82
p_inactive_given_nochurn = 0.05
p_inactive = (p_inactive_given_churn * p_prior +
              p_inactive_given_nochurn * (1 - p_prior))
p_posterior = (p_inactive_given_churn * p_prior) / p_inactive
print(f"P(churn | Premium, inactive 30d) = {p_posterior:.3f}")
print(f"(Prior was {p_prior:.3f} — the 30-day inactivity signal raises it to {p_posterior:.3f})")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import statistics
import math


tickets = [3, 5, 2, 4, 6, 3, 7, 2, 4, 5, 3, 6, 4, 2, 8, 3, 5, 4, 3, 6, 2, 4, 5, 3, 7, 4, 2, 5, 3, 4]

mean_t = statistics.mean(tickets)
med_t  = statistics.median(tickets)
std_t  = statistics.stdev(tickets)
var_t  = statistics.variance(tickets)

lam = mean_t
p_5 = (lam ** 5 * math.exp(-lam)) / math.factorial(5)

print(f"Mean:     {mean_t:.2f}")
print(f"Median:   {med_t:.2f}")
print(f"Std:      {std_t:.2f}")
print(f"Variance: {var_t:.2f}")
print(f"P(k=5 | λ={lam:.2f}): {p_5:.4f}")
print(f"Poisson-consistent? mean≈variance: {mean_t:.2f}≈{var_t:.2f} → "
      f"{'Yes' if abs(mean_t - var_t) < 1.0 else 'No'}")

In [ ]:
import math


# (a) Binomial
n, p = 320, 0.30
expected = n * p
std_b    = math.sqrt(n * p * (1 - p))
print(f"(a) Expected VN Free churners: {expected:.0f} ± {std_b:.1f}")

# (b) Bayes update
prior    = 0.30
p_inact_c  = 0.82
p_inact_nc = 0.08
p_inact  = p_inact_c * prior + p_inact_nc * (1 - prior)
posterior = (p_inact_c * prior) / p_inact
print(f"(b) P(churn | VN, Free, inactive 30d) = {posterior:.3f}")
print(f"    (Prior was {prior:.3f} — inactivity signal raises it to {posterior:.3f})")

---

# Chapter 66b: Statistical Inference & Experimentation

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy

### 2.1 The Central Limit Theorem — demonstrated

In [ ]:
import random
import statistics


random.seed(42)

# Underlying "population": subscription tenure in months
# Strongly right-skewed (most users are new; a few are long-tenured veterans)
population = [random.expovariate(1/12) for _ in range(10_000)]   # mean=12

pop_mean = statistics.mean(population)
pop_stdev = statistics.stdev(population)
print(f"Population: mean={pop_mean:.1f} months, std={pop_stdev:.1f} months")
print(f"(Strongly right-skewed — exponential distribution)")

# Draw 1000 samples of size 30, compute the sample mean each time
sample_means = []
for _ in range(1000):
    sample = random.choices(population, k=30)
    sample_means.append(statistics.mean(sample))

sm_mean  = statistics.mean(sample_means)
sm_stdev = statistics.stdev(sample_means)

print(f"\nSampling distribution of sample mean (n=30, 1000 draws):")
print(f"  Mean of sample means: {sm_mean:.1f}  (≈ population mean {pop_mean:.1f})")
print(f"  Std of sample means:  {sm_stdev:.2f}  (≈ std_error = {pop_stdev/30**0.5:.2f})")
print(f"  (CLT: sample mean is approx Normal even though population is exponential)")

### 2.2 Confidence Intervals

In [ ]:
import statistics
import math


# Sample of 40 watch_minutes values from a CinemaStream session
random_seed = 42
# Simulate a sample (in practice this would be real data)
import random
random.seed(random_seed)
sample = [random.gauss(54.2, 38.5) for _ in range(40)]   # mu=54.2, sigma=38.5

n    = len(sample)
mean = statistics.mean(sample)
se   = statistics.stdev(sample) / math.sqrt(n)

# 95% CI using z=1.96 (approximate — valid for n >= 30)
z = 1.96
ci_lower = mean - z * se
ci_upper = mean + z * se

print(f"Sample size: {n}")
print(f"Sample mean: {mean:.1f} minutes")
print(f"Std error:   {se:.2f}")
print(f"95% CI:      [{ci_lower:.1f}, {ci_upper:.1f}] minutes")
print(f"\nInterpretation: we estimate the true average session length")
print(f"is between {ci_lower:.0f} and {ci_upper:.0f} minutes, with 95% confidence.")

### 2.3 Hypothesis Testing — t-test

In [ ]:
from scipy import stats


# Are Premium and Basic subscribers watching more minutes per session?
import random
random.seed(42)

# Simulate session lengths for two groups
premium_sessions = [random.gauss(68.3, 35.1) for _ in range(80)]
basic_sessions   = [random.gauss(52.4, 38.2) for _ in range(120)]

import statistics
print(f"Premium: n={len(premium_sessions)}, mean={statistics.mean(premium_sessions):.1f} min")
print(f"Basic:   n={len(basic_sessions)},  mean={statistics.mean(basic_sessions):.1f} min")
print(f"Observed difference: {statistics.mean(premium_sessions) - statistics.mean(basic_sessions):.1f} min")

# Two-sample t-test (Welch's t-test — doesn't assume equal variances)
t_stat, p_value = stats.ttest_ind(premium_sessions, basic_sessions, equal_var=False)
print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value:     {p_value:.4f}")

alpha = 0.05
conclusion = "Reject H0" if p_value < alpha else "Fail to reject H0"
print(f"\nAt α={alpha}: {conclusion}")
if p_value < alpha:
    print("→ Evidence that Premium subscribers watch significantly more per session.")
else:
    print("→ Difference could be due to chance; insufficient evidence.")

### 2.4 Chi-square test — for categorical variables

In [ ]:
from scipy import stats
import numpy as np


# Question: is churn rate independent of plan type?
# Observed counts from CinemaStream's 100-subscriber dataset (canonical Ch060)
#              Churned  Not churned
# Free           17        39
# Basic           5        23
# Premium         2        14

observed = np.array([
    [17, 39],   # Free
    [ 5, 23],   # Basic
    [ 2, 14],   # Premium
])

chi2, p_value, dof, expected = stats.chi2_contingency(observed)

print(f"Chi-square statistic: {chi2:.3f}")
print(f"Degrees of freedom:   {dof}")
print(f"p-value:              {p_value:.4f}")
print(f"\nExpected counts (under H0: churn is independent of plan):")
print(f"  Free:     churned={expected[0,0]:.1f}, not={expected[0,1]:.1f}")
print(f"  Basic:    churned={expected[1,0]:.1f}, not={expected[1,1]:.1f}")
print(f"  Premium:  churned={expected[2,0]:.1f}, not={expected[2,1]:.1f}")

alpha = 0.05
print(f"\nAt α={alpha}: {'Reject H0' if p_value < alpha else 'Fail to reject H0'}")
if p_value < alpha:
    print("→ Churn rate IS associated with plan type.")
else:
    print("→ No significant association detected.")

### 2.5 A/B Test Design — power and sample size

In [ ]:
import math


def sample_size_per_group(
    baseline: float,
    mde: float,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """
    Estimate required n per group for a two-proportion z-test.
    Uses the standard formula with two-tailed z-values.
    """
    p1 = baseline
    p2 = baseline + mde
    p_bar = (p1 + p2) / 2

    # z-scores for alpha/2 (two-tailed) and power
    # alpha=0.05 -> z_alpha=1.96; power=0.80 -> z_beta=0.842
    # Use approximation: Phi^{-1}(0.975) = 1.96, Phi^{-1}(0.80) = 0.842
    z_alpha = 1.96
    z_beta  = 0.842

    numerator   = (z_alpha * math.sqrt(2 * p_bar * (1 - p_bar))
                   + z_beta * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    denominator = (p2 - p1) ** 2
    return math.ceil(numerator / denominator)


# CinemaStream: 7-day retention, testing a new onboarding flow
baseline = 0.62   # current 7-day retention rate
scenarios = [
    ("Detect +3pp  (ambitious)", 0.03),
    ("Detect +5pp  (realistic)", 0.05),
    ("Detect +10pp (conservative)", 0.10),
]
print(f"Baseline retention: {baseline:.0%}")
print(f"{'Scenario':<35} {'n/group':>8}  {'Total n':>8}  {'~Days @ 2k/day':>14}")
print("-" * 70)
for label, mde in scenarios:
    n = sample_size_per_group(baseline, mde)
    days = math.ceil((2 * n) / 2000)
    print(f"{label:<35} {n:>8,}  {2*n:>8,}  {days:>14}")

## 3. CinemaStream in Practice

In [ ]:
import math
from scipy import stats
import random


# --- Step 1: Design the experiment ---
baseline_retention  = 0.62   # current 7-day retention, Free plan
target_mde          = 0.05   # Rohan: "I want to detect at least a 5pp lift"
alpha               = 0.05
power               = 0.80


def sample_size_two_proportions(p1, mde, alpha=0.05, power=0.80):
    p2    = p1 + mde
    p_bar = (p1 + p2) / 2
    z_a   = 1.96
    z_b   = 0.842
    num   = (z_a * math.sqrt(2 * p_bar * (1 - p_bar))
             + z_b * math.sqrt(p1*(1-p1) + p2*(1-p2))) ** 2
    return math.ceil(num / (mde ** 2))


n_per_group = sample_size_two_proportions(baseline_retention, target_mde)
new_signups_per_day = 180   # Free-plan new sign-ups per day (illustrative)
days_needed = math.ceil((2 * n_per_group) / new_signups_per_day)

print("=== Experiment Design ===")
print(f"Baseline retention:    {baseline_retention:.0%}")
print(f"Min detectable effect: +{target_mde:.0%}")
print(f"α={alpha}, power={power}")
print(f"Required n per group:  {n_per_group:,}")
print(f"Total participants:    {2 * n_per_group:,}")
print(f"Run time (at {new_signups_per_day}/day): {days_needed} days (~{days_needed//7} weeks)")

# --- Step 2: Simulate the experiment outcome ---
random.seed(42)
p_control  = 0.62
p_treatment = 0.67   # true lift of +5pp (we get to set this in simulation)

control   = [1 if random.random() < p_control  else 0 for _ in range(n_per_group)]
treatment = [1 if random.random() < p_treatment else 0 for _ in range(n_per_group)]

n_c, retain_c  = len(control),   sum(control)
n_t, retain_t  = len(treatment), sum(treatment)

print(f"\n=== Observed Results (after {days_needed} days) ===")
print(f"Control:   {retain_c}/{n_c} retained  ({retain_c/n_c:.1%})")
print(f"Treatment: {retain_t}/{n_t} retained  ({retain_t/n_t:.1%})")
print(f"Observed lift: +{(retain_t/n_t - retain_c/n_c):.1%}")

# --- Step 3: Test for significance ---
_, p_value = stats.ttest_ind(treatment, control)
print(f"\n=== Statistical Test ===")
print(f"p-value: {p_value:.4f}")
print(f"Significant at α=0.05: {'Yes — ship it' if p_value < alpha else 'No — do not ship'}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import statistics
from scipy import stats


group_a = [120, 132, 115, 128, 140, 118, 135, 122, 129, 131]
group_b = [135, 148, 129, 141, 155, 133, 149, 138, 144, 147]

print(f"Group A mean: S${statistics.mean(group_a):.1f}")
print(f"Group B mean: S${statistics.mean(group_b):.1f}")
print(f"Difference:   S${statistics.mean(group_b) - statistics.mean(group_a):.1f}")

t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)
print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value:     {p_value:.4f}")
print(f"Significant at α=0.05: {'Yes' if p_value < 0.05 else 'No'}")

In [ ]:
import math


def sample_size_two_proportions(p1, mde, alpha=0.05, power=0.80):
    p2    = p1 + mde
    p_bar = (p1 + p2) / 2
    z_a   = 1.96
    z_b   = 0.842
    num   = (z_a * math.sqrt(2 * p_bar * (1 - p_bar))
             + z_b * math.sqrt(p1*(1-p1) + p2*(1-p2))) ** 2
    return math.ceil(num / (mde ** 2))


p1  = 0.048    # current upgrade rate
mde = 0.022    # target lift: 4.8% → 7.0%

n = sample_size_two_proportions(p1, mde)
daily_signups = 600
days = math.ceil((2 * n) / daily_signups)

print(f"Baseline upgrade rate: {p1:.1%}")
print(f"Target rate:           {p1 + mde:.1%}")
print(f"MDE:                   +{mde:.1%}")
print(f"Required n per group:  {n:,}")
print(f"Total participants:    {2*n:,}")
print(f"Days to run:           {days} (~{days//7} weeks)")

---

# Chapter 66c: Linear Algebra, Embeddings & Optimization for ML

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy

### 2.1 Vectors — the basics

In [ ]:
import numpy as np


# A user's feature vector: [watch_minutes_avg, tenure_months, support_tickets]
user_ravi  = np.array([132.0, 18.0, 1.0])
user_minh  = np.array([ 23.0,  4.0, 3.0])
user_siti  = np.array([ 96.0, 28.0, 0.0])

print("User vectors:")
print(f"  Ravi:  {user_ravi}")
print(f"  Minh:  {user_minh}")
print(f"  Siti:  {user_siti}")

# Vector arithmetic
print("\nVector addition (Ravi + Minh):")
print(f"  {user_ravi + user_minh}")

print("\nScalar multiplication (Ravi × 2):")
print(f"  {user_ravi * 2}")

# Vector magnitude (L2 norm)
def magnitude(v):
    return float(np.sqrt(np.dot(v, v)))

print("\nMagnitudes (L2 norm):")
for name, v in [("Ravi", user_ravi), ("Minh", user_minh), ("Siti", user_siti)]:
    print(f"  ||{name}|| = {magnitude(v):.2f}")

### 2.2 Dot product and cosine similarity

In [ ]:
import numpy as np


def dot_product(a, b):
    return float(np.dot(a, b))

def cosine_similarity(a, b):
    return dot_product(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# RAW user feature vectors — deliberately unscaled, and not trained embeddings.
# Their norms are ~133, ~24 and ~100 (printed in 2.1), not 1.0. That is the point:
# these are the raw features that expose the scaling problem discussed below.
user_ravi  = np.array([132.0, 18.0, 1.0])
user_minh  = np.array([ 23.0,  4.0, 3.0])
user_siti  = np.array([ 96.0, 28.0, 0.0])

pairs = [
    ("Ravi", "Siti",  user_ravi, user_siti),
    ("Ravi", "Minh",  user_ravi, user_minh),
    ("Siti", "Minh",  user_siti, user_minh),
]

print("Cosine similarities between users:")
for n1, n2, v1, v2 in pairs:
    sim = cosine_similarity(v1, v2)
    print(f"  sim({n1}, {n2}) = {sim:.4f}")

print("\nInterpretation:")
print("  sim near 1.0 → vectors point in the same direction → similar users")
print("  sim near 0.0 → vectors are perpendicular → dissimilar behaviour")

### 2.3 Matrix multiplication — the workhorse

In [ ]:
import numpy as np


# Dataset: 4 users × 3 features
X = np.array([
    [132.0, 18.0, 1.0],   # Ravi
    [ 23.0,  4.0, 3.0],   # Minh
    [ 96.0, 28.0, 0.0],   # Siti
    [ 45.0,  7.0, 2.0],   # generic user 4
])
print(f"X shape: {X.shape}  (4 users × 3 features)")

# Weight matrix: 3 input features → 2 output features (a simple linear layer)
W = np.array([
    [0.5, -0.1],   # weight row for feature 0 (watch_minutes)
    [0.3,  0.8],   # weight row for feature 1 (tenure)
    [-0.2, 0.4],   # weight row for feature 2 (tickets)
])
print(f"W shape: {W.shape}  (3 features → 2 outputs)")

# Matrix multiplication: (4, 3) @ (3, 2) → (4, 2)
Z = X @ W
print(f"Z = X @ W shape: {Z.shape}  (4 users × 2 outputs)\n")
print("Output Z (transformed features):")
for i, (name, row) in enumerate(zip(["Ravi", "Minh", "Siti", "User4"], Z)):
    print(f"  {name}: [{row[0]:.2f}, {row[1]:.2f}]")

# Shape rule: (m, k) @ (k, n) → (m, n)
# The inner dimensions MUST match
print(f"\nShape rule: ({X.shape[0]}, {X.shape[1]}) @ ({W.shape[0]}, {W.shape[1]}) "
      f"→ ({X.shape[0]}, {W.shape[1]})")
print("The middle dimension (3) must match — this is what 'shapes not aligned' means.")

### 2.4 Meaning as geometry — the embedding intuition

In [ ]:
import numpy as np


# Toy 2D embeddings (pretend these were learned from watch history)
# Each movie gets a 2D vector: [action_score, drama_score]
movies = {
    "Monsoon Heart":       np.array([0.1, 0.95]),   # pure drama
    "Hujan di Singapura":  np.array([0.2, 0.85]),   # drama with tension
    "Jakarta Drift":       np.array([0.95, 0.1]),   # pure action
    "Tigers at Dawn":      np.array([0.85, 0.2]),   # action with story
    "The Quiet Monsoon":   np.array([0.15, 0.80]),  # drama
}

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Query: user just watched "Monsoon Heart" — what to recommend?
query = movies["Monsoon Heart"]
print("Recommendations for user who watched 'Monsoon Heart':")
print(f"{'Movie':<25} {'Cosine Sim':>12}")
print("-" * 40)
for title, vec in movies.items():
    if title == "Monsoon Heart":
        continue
    sim = cosine_sim(query, vec)
    print(f"  {title:<23} {sim:.4f}")

print("\nMeaning captured by geometry:")
print("  Drama movies cluster together → high mutual similarity")
print("  Action movies cluster together → high mutual similarity")
print("  Drama vs Action → low similarity (perpendicular directions)")

### 2.5 Gradient descent — optimization as geometry

In [ ]:
import numpy as np


# Simple 1D example: minimize f(w) = (w - 3)^2
# Minimum is at w = 3 (where f(w) = 0)
# Gradient: df/dw = 2(w - 3)

def f(w):
    return (w - 3) ** 2

def gradient_f(w):
    return 2 * (w - 3)

# Gradient descent
w        = 0.0     # start far from the minimum
lr       = 0.1     # learning rate
n_steps  = 20

print(f"{'Step':>5}  {'w':>8}  {'f(w)':>8}  {'grad':>8}")
print("-" * 38)
for step in range(n_steps):
    loss = f(w)
    grad = gradient_f(w)
    w    = w - lr * grad                   # gradient descent update
    if step < 8 or step >= 18:
        print(f"{step:>5}  {w:>8.4f}  {loss:>8.4f}  {grad:>8.4f}")
    elif step == 8:
        print(f"  ... (converging) ...")

print(f"\nFinal w = {w:.4f}  (true minimum: 3.0000)")
print(f"Final loss = {f(w):.8f}")

## 3. CinemaStream in Practice

In [ ]:
import numpy as np


# --- User-movie interaction matrix (watch_minutes) ---
# 5 users × 5 movies (rows = users, columns = movies)
# movie_ids: 101=Monsoon Heart, 102=Hujan di Singapura,
#            103=Jakarta Drift, 104=Tigers at Dawn, 105=The Quiet Monsoon

user_movie_matrix = np.array([
    #  101   102   103   104   105
    [132,  89,   0,    0,   45],   # user 1 (Ravi — drama watcher)
    [  0,   0,  98,  120,   0],    # user 2 (Siti — action watcher)
    [ 23,  45,   0,    0,  67],    # user 3 (Minh — casual drama)
    [  0,  12,  78,   90,   0],    # user 4 (action leaning)
    [ 56,   0,   0,   23,  88],    # user 5 (drama + some action)
], dtype=float)

print("User-Movie interaction matrix:")
movie_names = ["Monsoon", "Hujan", "Jakarta", "Tigers", "Quiet M"]
print(f"{'':>8} " + " ".join(f"{n:>8}" for n in movie_names))
for i, row in enumerate(user_movie_matrix):
    print(f"User {i+1:2d}: " + " ".join(f"{v:>8.0f}" for v in row))

# --- Item similarity: which movies are most similar in terms of who watches them? ---
# Normalize columns (movie vectors) to unit length
movie_matrix = user_movie_matrix.T   # shape: (5 movies × 5 users)
norms = np.linalg.norm(movie_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1e-9             # avoid division by zero
movie_vecs = movie_matrix / norms

# Cosine similarity matrix: (5 × 5)
sim_matrix = movie_vecs @ movie_vecs.T

print("\nMovie-Movie cosine similarity:")
print(f"{'':>10} " + " ".join(f"{n:>10}" for n in movie_names))
for i, name in enumerate(movie_names):
    row_str = " ".join(f"{sim_matrix[i,j]:>10.3f}" for j in range(len(movie_names)))
    print(f"{name:>10} {row_str}")

# --- Recommendation: user 3 (Minh) watched dramas; what to suggest next? ---
# Item-based collaborative filtering. Score a candidate movie j by how similar it
# is to the movies Minh actually watched, weighted by how much he watched them:
#     score(j) = Σ_i  minh_watch[i] × sim(i, j)
# Both i and j are MOVIE indices, so the dimensions line up. (Dotting Minh's row —
# indexed by movie — against a column of movie_matrix — indexed by user — would
# also "work" in NumPy, because both happen to be length 5, and would be
# meaningless. Matching lengths are not matching dimensions.)
minh_watch      = user_movie_matrix[2]                              # minutes per movie
already_watched = {j for j, mins in enumerate(minh_watch) if mins > 0}
scores = {}
for j in range(len(movie_names)):
    if j not in already_watched:
        scores[movie_names[j]] = float(sum(minh_watch[i] * sim_matrix[i, j]
                                           for i in already_watched))

print(f"\nRecommendations for Minh (user 3, drama watcher):")
print(f"  (watched: {', '.join(movie_names[i] for i in sorted(already_watched))})")
for title, score in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"  {title}: score={score:.1f}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import numpy as np


a = np.array([3, 1, 4, 1, 5], dtype=float)
b = np.array([2, 7, 1, 8, 2], dtype=float)

dot   = float(np.dot(a, b))
mag_a = float(np.linalg.norm(a))
mag_b = float(np.linalg.norm(b))
cos   = dot / (mag_a * mag_b)

print(f"Dot product:       {dot:.2f}")
print(f"||a||:             {mag_a:.4f}")
print(f"||b||:             {mag_b:.4f}")
print(f"Cosine similarity: {cos:.4f}")
print(f"Interpretation: {'similar (cos > 0.7)' if cos > 0.7 else 'dissimilar (cos < 0.7)'}")

In [ ]:
import numpy as np
import math


w = np.array([0.4, -0.3, 0.5, -0.1, 0.6, 0.2, -0.2, 0.3])
x = np.array([85.2, 12.0, 14.0, 2.0, 5.8, 0.0, 1.0, 0.0])

raw_score = float(np.dot(w, x))
sigmoid   = 1 / (1 + math.exp(-raw_score))

print(f"Raw score (w · x): {raw_score:.4f}")
print(f"Sigmoid(score):    {sigmoid:.4f}")
print(f"Threshold=0.3: {'CHURN FLAG' if sigmoid > 0.3 else 'No flag'}")